=============================================================================
PIPELINE 1 OPTIMIZED — ESG Document Filtering V10
=============================================================================

ARCHITECTURE CHANGE FROM V9:
  V9:  One filter function per document type
  V10: Document-type detection → routes to specialized sub-filters

  Sustainability reports now use 3 strategies:

    1. Standard reports → keyword + data-table detection (improved V9)
    2. CDP questionnaires → CDP section ID matching (C6.1, C7.9, etc.)
    3. Website-to-PDF → loose keyword + structural analysis

  DEF 14A now uses:

    - Broad regex patterns for phrasing variation
    - Optional lightweight API classification (Haiku, ~$0.007/doc)


KEY FIXES FOR RECALL:

  - CDP: Section ID matching instead of keyword matching
  - Website PDFs: Adapted extraction that doesn't rely on pdfplumber tables
  - DEF 14A: Regex patterns catch "CEO Pay Rate", "Pay Rate",
    "Chief Executive Officer Compensation", footnote markers, etc.
  - All types: Adjacent page inclusion for multi-page data tables
  - Exclusion patterns prevent wasting page budget on boilerplate


INTEGRATION:

  - Replace cells 12-18 (config + keywords) and cells 25-27 (filter functions)
  - in your Colab notebook with this code. Update cell 33 (main pipeline) to
  - call the new function names or rename them.


In [1]:
# =============================================================================
# CELL 1: GOOGLE COLAB SETUP
# =============================================================================

# If force_remount not working
# try:
#     drive.flush_and_unmount()
# except:
#     pass

# # Remove the directory completely
# !rm -rf /content/drive


from google.colab import drive
drive.mount('/content/drive', force_remount=True) # , force_remount=True

print("✅ Google Drive mounted")

Mounted at /content/drive
✅ Google Drive mounted


In [2]:

# =============================================================================
# CELL 2: INSTALL DEPENDENCIES
# =============================================================================

!pip install pymupdf pdfplumber -q

print("✅ Dependencies installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 94.2 MB/s eta 0:00:00
✅ Dependencies installed


In [3]:
# =============================================================================
# CELL 3: IMPORTS
# =============================================================================

import os
import re
import json
import shutil
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Set
from datetime import datetime
from collections import defaultdict

import fitz  # PyMuPDF
import pdfplumber

print("✅ Imports complete")

✅ Imports complete


In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# Google Drive paths
DRIVE_BASE = Path("/content/drive/MyDrive")

# Input: Original company documents
INPUT_DIR = DRIVE_BASE / "ESG Data" / "reports"

# Output: Filtered documents
OUTPUT_DIR = DRIVE_BASE / "ESG Data" / "filtered_documents_4"  # versiom 4 is the optimal fitlering config
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Filtering log
LOG_DIR = DRIVE_BASE / "ESG Data" / "filtering_logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Years to process
YEARS = [2020, 2021, 2022, 2023, 2024]

In [ ]:
# =============================================================================
# COMPANY LIST
# =============================================================================

TEST_COMPANIES = [
    "Dell_Technologies",
    "HP_Inc"
]

# Full company list
ALL_COMPANIES = [
    "Accenture", "Adobe_Inc", "Advanced_Micro_Devices", "Akamai_Technologies",
    "Amphenol", "Analog_Devices", "Apple_Inc", "Applied_Materials", "Arista_Networks",
    "Autodesk", "Broadcom", "Cadence_Design_Systems", "CDW_Corporation", "Cisco",
    "Cognizant", "Corning_Inc", "CrowdStrike", "Datadog", "Dell_Technologies",
    "Enphase_Energy", "EPAM_Systems", "F5_Inc", "Fair_Isaac", "First_Solar",
    "Fortinet", "Gartner", "Gen_Digital", "GoDaddy", "Hewlett_Packard_Enterprise",
    "HP_Inc", "IBM", "Intel", "Intuit", "Jabil", "Keysight_Technologies",
    "KLA_Corporation", "Lam_Research", "Microchip_Technology", "Micron_Technology",
    "Microsoft", "Monolithic_Power_Systems", "Motorola_Solutions", "NetApp", "Nvidia",
    "NXP_Semiconductors", "ON_Semiconductor", "Oracle_Corporation", "Palantir_Technologies",
    "Palo_Alto_Networks", "PTC_Inc", "Qualcomm", "Roper_Technologies", "Salesforce",
    "Seagate_Technology", "ServiceNow", "Skyworks_Solutions", "Supermicro", "Synopsys",
    "TE_Connectivity", "Teledyne_Technologies", "Teradyne", "Texas_Instruments",
    "Trimble_Inc", "Tyler_Technologies", "Verisign", "Western_Digital", "Workday_Inc",
    "Zebra_Technologies"
]

print(f"✅ {len(ALL_COMPANIES)} companies in full list")
print(f"   Testing with: {TEST_COMPANIES}")

✅ 68 companies in full list
   Testing with: ['Dell_Technologies', 'HP_Inc']


## Filtering Parameters

### Document Config

In [45]:
# =============================================================================
# FILTERING PARAMETERS - V10
# =============================================================================

SUSTAINABILITY_CONFIG = {
    # Standard report settings
    "top_pages_per_issue": 2,          # Max pages per issue (adaptive picks fewer)            # Test 4 setting
    "min_issue_score": 2,              # Minimum keyword matches to qualify
    "top_table_indicator_pages": 3,    # Extra table-indicator pages (non-overlapping only)    # Test 4 setting
    "max_pages_to_select": 50,         # Safety cap
    "score_gap_ratio": 0.25,           # Stop selecting if page scores < 25% of top page
    "adjacent_page_threshold": 0.4,    # Include ±1 page if it scores ≥ 40% of min score

    # CDP questionnaire settings
    "cdp_context_pages": 1,            # Pages before/after a CDP match to include

    # Website-to-PDF settings
    "web_min_keywords_per_page": 1,    # Lower threshold for website docs
    "web_top_pages_per_issue": 4,      # More generous for noisy web extractions
}

DEF14A_CONFIG = {
    "top_pages_per_category": 3,
    "min_category_score": 1,           # Keep at 1 — regex patterns are already precise
    "max_pages_to_select": 30,
    "adjacent_page_threshold": 0.4,
    "use_api_classification": False,   # Set True to enable Haiku classification
    "api_model": "claude-haiku-4-5-20251001",
}

# 10-K as HTML
# TENK_CONFIG = {
#     "max_sections": 30,
#     "min_section_length": 50,
# }

# 10-K as PDF
TENK_CONFIG = {
    "top_pages_per_group": 3,          # Top pages per keyword group
    "min_group_score": 1,              # Minimum score to qualify
    "max_pages_to_select": 30,         # Safety cap
    "score_gap_ratio": 0.25,           # Stop selecting if score < 25% of best
    "adjacent_page_threshold": 0.4,    # Include ±1 page if scores above threshold
}


IMAGE_FALLBACK_CONFIG = {
    "max_pdf_size_mb": 15.0,
    "target_width": 1400,
    "jpeg_quality": 85,
}

# Most impactful parameters:

# 1. max_pages_to_select (sustainability) — Currently 50.
# This is your hard cap and several documents are hitting it.
# Try lowering to 35 or 40. This is the bluntest lever but also
# the most effective for the documents that are ballooning.

# 2. top_pages_per_issue (sustainability) — Currently 3.
# With 17 issues × 3 pages = up to 51 pages just from issue selection.
# Dropping to 2 would cap issue pages at 34.
# This is the single most impactful parameter since most issues only need 1–2 pages.
# The score gap ratio already limits over-selection, but capping at 2 adds a hard guarantee.

# 3. score_gap_ratio (sustainability) — Currently 0.25.
# This means a page scoring 25% of the best page still gets selected.
# Raising to 0.4 or 0.5 would be more selective — a page needs to score
# at least 40–50% of the top page to qualify.
# This cuts low-confidence pages without affecting strong matches.

# 4. top_table_indicator_pages — Currently 5.
# These are extra pages beyond the per-issue selection. Try 3.
# If a table indicator page is genuinely important, it's likely already
# selected by an issue.

# 5. adjacent_page_threshold — Currently 0.4.
# Raising to 0.6 or 0.7 means adjacent pages need stronger evidence to be included.
# Or set to 0 to disable adjacent page inclusion entirely and see how recall is affected —
# you might not need it for standard reports.

# 6. top_pages_per_category (DEF 14A) — Currently 3.
# Try 2 for categories like board, meetings, and ethics where the data is usually
# on 1–2 pages. You could keep 3 for pay_ratio and say_on_pay since those results
# can span pages.

print(f"✅ Configuration loaded")
print(f"   Input: {INPUT_DIR}")
print(f"   Output: {OUTPUT_DIR}")
print(f"   Logs: {LOG_DIR}")

✅ Configuration loaded
   Input: /content/drive/MyDrive/ESG Data/reports
   Output: /content/drive/MyDrive/ESG Data/filtered_documents_4
   Logs: /content/drive/MyDrive/ESG Data/filtering_logs


### Document Type Detection

In [46]:
# =============================================================================
# DOCUMENT TYPE DETECTION + EXCLUSION PATTERNS + HELPER FUNCTIONS
# =============================================================================

class DocType:
    STANDARD = "standard"       # Regular sustainability report PDF
    CDP = "cdp"                 # CDP questionnaire
    WEB_PDF = "web_pdf"         # Website downloaded as PDF


def detect_sustainability_doc_type(pdf_path: str, filename: str) -> str:
    """
    Detect what kind of sustainability document this is.
    1. Check filename for hints (user can rename CDPs and web PDFs)
    2. Check first 5 pages for structural signals
    """
    filename_lower = filename.lower()

    # Filename-based detection
    if any(tag in filename_lower for tag in ['cdp', 'questionnaire', 'climate_response']):
        return DocType.CDP
    if any(tag in filename_lower for tag in ['web', 'website', 'webpage']):
        return DocType.WEB_PDF

    # Content-based detection (first 5 pages)
    try:
        with pdfplumber.open(pdf_path) as pdf:
            first_pages_text = ""
            for i, page in enumerate(pdf.pages[:5]):
                text = page.extract_text() or ""
                first_pages_text += text.lower() + "\n"

        cdp_signals = [
            'cdp climate change', 'cdp questionnaire', 'cdp response',
            'climate change 20', 'c0.1', 'c0.2', 'c1.1', 'c1.2',
            'cdp water security', 'cdp forests',
        ]
        if sum(1 for s in cdp_signals if s in first_pages_text) >= 2:
            return DocType.CDP

        web_signals = [
            'cookie', 'privacy policy', 'terms of use', 'navigation',
            'skip to content', 'subscribe', 'footer', 'breadcrumb',
            'www.', '.com/', 'http://', 'https://',
        ]
        if sum(1 for s in web_signals if s in first_pages_text) >= 3:
            return DocType.WEB_PDF

    except Exception:
        pass

    return DocType.STANDARD


# ── Exclusion patterns ──

SUSTAINABILITY_EXCLUSION_KEYWORDS = [
    #'table of contents',
    'about this report',
    'this report contains forward-looking',
    'forward-looking statements',
    'stakeholder engagement process',
    'ceo letter', 'ceo message', 'letter from our ceo',
    'letter to our stakeholders',
    'dear stakeholders', 'dear shareholders',
    'report overview',
    'gri content index',
    'un sustainable development goals',
    'sdg alignment',
    'legal disclaimer',
    'cautionary statement',
    'safe harbor',
]

DEF14A_EXCLUSION_KEYWORDS = [
    # NOTE: 'table of contents' is deliberately NOT here.
    # SEC EDGAR proxy filings have "Table of Contents" as a running
    # header on EVERY page — including it kills the entire document.
    # Same for 'forward-looking statements' and 'cautionary statement'.
    'how to vote your shares',
    'voting instructions',
    'questions and answers about the meeting',
    'internet availability of proxy materials',
    'important notice regarding the availability',
]


# ── Helper functions ──

def get_exclusion_penalty(text_lower: str, exclusion_keywords: list) -> int:
    """Check if a page matches exclusion patterns. Returns negative penalty."""
    matched = [kw for kw in exclusion_keywords if kw in text_lower]
    if not matched:
        return 0
    penalty = -15 * len(matched)
    return max(penalty, -100)


def is_data_table(table) -> Tuple[bool, int]:
    """
    Determine if a table is a genuine data table vs TOC/layout table.
    Returns (is_data, quality_score).
    """
    if not table or len(table) < 2:
        return False, 0

    numeric_rows = 0
    total_numeric_cells = 0

    for row in table:
        if not row:
            continue
        row_has_number = False
        for cell in row:
            cell_str = str(cell) if cell else ''
            if re.search(r'\b\d{1,3}(?:,\d{3})+\b|\b\d+\.\d+\b|\d+%', cell_str):
                total_numeric_cells += 1
                row_has_number = True
        if row_has_number:
            numeric_rows += 1

    if numeric_rows < 2:
        return False, 0

    header_str = str(table[0]).lower() if table[0] else ''
    has_year_headers = sum(1 for y in YEAR_KEYWORDS if y in header_str)

    quality = 0
    if numeric_rows >= 3 and total_numeric_cells >= 4:
        quality = 15
    elif numeric_rows >= 2:
        quality = 8

    if has_year_headers >= 2:
        quality += 10
    if any(u in str(table).lower() for u in ['tco2e', 'mwh', '%', 'tonnes', 'hours']):
        quality += 5

    return quality > 0, quality


def count_data_numbers(text: str) -> int:
    """Count meaningful data numbers on a page (not page/footnote numbers)."""
    comma_nums = len(re.findall(r'\b\d{1,3}(?:,\d{3})+\b', text))
    decimals = len(re.findall(r'\b\d+\.\d+\b', text))
    percentages = len(re.findall(r'\b\d+(?:\.\d+)?%', text))
    large_ints = len(re.findall(r'\b\d{3,}\b', text))
    return comma_nums + decimals + percentages + large_ints


def add_adjacent_pages(selected: Set[int], page_scores: list,
                       total_pages: int, threshold: float) -> List[int]:
    """Add ±1 adjacent pages if they score above threshold."""
    adjacent_added = []
    snapshot = set(selected)
    for page_num in snapshot:
        for adj in [page_num - 1, page_num + 1]:
            if adj < 1 or adj > total_pages or adj in selected:
                continue
            adj_page_scores = [p for p in page_scores if p['page_num'] == adj]
            if adj_page_scores:
                best = max(s['score'] for s in adj_page_scores)
                if best >= threshold:
                    selected.add(adj)
                    adjacent_added.append(adj)
    return adjacent_added


def suggest_renames(company_dir: Path) -> Dict[str, str]:
    """
    Scan a company's documents and suggest renames for CDP/web PDFs.
    Returns dict of {current_name: suggested_name}.
    Run this once to identify which files need renaming.
    """
    suggestions = {}
    for pdf_path in company_dir.glob("*.pdf"):
        if any(x in pdf_path.name for x in ['10-K', '10K', 'DEF_14A', 'DEF14A']):
            continue
        try:
            with pdfplumber.open(str(pdf_path)) as pdf:
                first_pages = ""
                for page in pdf.pages[:3]:
                    first_pages += (page.extract_text() or "").lower() + "\n"
            doc_type = "standard"
            if sum(1 for s in ['cdp', 'c0.1', 'c1.1', 'climate change 20'] if s in first_pages) >= 2:
                doc_type = "cdp"
            elif sum(1 for s in ['cookie', 'privacy policy', 'skip to content', 'subscribe'] if s in first_pages) >= 2:
                doc_type = "web"
            if doc_type != "standard":
                new_name = pdf_path.name.replace("Report_", f"Report_{doc_type.upper()}_")
                suggestions[pdf_path.name] = new_name
                print(f"  {doc_type.upper()}: {pdf_path.name} → {new_name}")
        except Exception:
            continue
    return suggestions


print("✅ Doc type detection, exclusion patterns, and helper functions defined")

✅ Doc type detection, exclusion patterns, and helper functions defined


### Keywords


In [47]:
# =============================================================================
# EXCLUSION PATTERNS
# =============================================================================

SUSTAINABILITY_EXCLUSION_KEYWORDS = [
    #'table of contents',
    'about this report',
    'this report contains forward-looking',
    'forward-looking statements',
    'stakeholder engagement process',
    'ceo letter', 'ceo message', 'letter from our ceo',
    'letter to our stakeholders',
    'dear stakeholders', 'dear shareholders',
    'report overview',
    #'gri content index',           # Cross-reference table, not actual data
    'un sustainable development goals',
    'sdg alignment',
    'legal disclaimer',
    'cautionary statement',
    'safe harbor',
]

DEF14A_EXCLUSION_KEYWORDS = [
    #'table of contents',
    'forward-looking statements',
    'cautionary statement',
    'notice of annual meeting',
    'how to vote your shares',
    'voting instructions',
    'questions and answers about the meeting',
    'internet availability of proxy materials',
    'important notice regarding',
]

In [48]:
# =============================================================================
# YEAR AND UNIT KEYWORDS
# =============================================================================

YEAR_KEYWORDS = ['2020', '2021', '2022', '2023', '2024']

UNIT_KEYWORDS = [
    'tonnes', 'metric tons', 'kilograms', 'megawatts', 'gigajoules',
    'per employee', 'per capita', 'per revenue', 'intensity', 'tco2e',
    'mwh', 'gwh', 'kwh', 'm³', 'cubic meters', 'megaliters', 'gallons',
    'per million', 'per 200,000', 'per 1,000,000',
]

In [49]:
# =============================================================================
# SUSTAINABILITY KEYWORDS (for standard reports)
# =============================================================================

SUSTAINABILITY_ISSUE_KEYWORDS = {
    # ENVIRONMENT - Climate Change
    'carbon_emissions': [
        'scope 1', 'scope 2', 'scope 3', 'tco2e', 'co2e', 'mtco2e', 'ktco2e',
        'greenhouse gas', 'ghg emissions', 'carbon emissions', 'direct emissions',
        'indirect emissions', 'emissions inventory', 'carbon footprint',
        'market-based', 'location-based', 'ghg protocol', 'carbon intensity',
        'absolute emissions', 'emissions reduction', 'ghg',
    ],
    'energy': [
        'energy consumption', 'total energy', 'renewable energy',
        'mwh', 'gwh', 'kwh', 'megawatt', 'gigawatt', 'energy intensity',
        'solar energy', 'wind power', 'clean energy', 'green energy',
        'renewable electricity', 'energy use', 'electricity consumption',
        'power purchase agreement', 'ppa', 'electricity',
    ],

    # ENVIRONMENT - Natural Capital
    'water': [
        'water withdrawal', 'water consumption', 'water usage', 'water intake',
        'cubic meters', 'megaliter', 'water stress', 'water recycled',
        'water intensity', 'freshwater', 'water discharge', 'water reuse',
        'water replenishment',
    ],

    # ENVIRONMENT - Pollution & Waste
    'waste': [
        'waste generated', 'total waste', 'waste recycled', 'hazardous waste',
        'non-hazardous', 'landfill', 'waste diverted', 'recycling rate',
        'circular economy', 'waste to landfill', 'waste disposal',
        'waste diversion', 'zero waste',
    ],
    'ewaste': [
        'e-waste', 'electronic waste', 'ewaste', 'electronics recycling',
        'it equipment disposal', 'end of life electronics',
        'end-of-life', 'product take-back', 'takeback',
    ],

    # ENVIRONMENT - Green Building
    'green_building': [
        'leed', 'green building', 'certified building', 'energy star',
        'sustainable building', 'building certification',
    ],

    # SOCIAL - Health & Safety
    'health_safety': [
        'trir', 'ltifr', 'lost time', 'lost-time', 'recordable injuries', 'injury rate', 'incident rate'
        'fatality', 'fatalities', 'workplace safety', 'occupational health',
        'safety incidents', 'work-related injuries', 'days away', 'dart rate',
        'safety performance', 'incident rate', 'total recordable', "well-being",
        'recordable injury', 'rate of recordable injury', 'lti'
    ],

    # SOCIAL - Labor / Workforce
    'workforce': [
        'total employees', 'employee headcount', 'number of employees',
        'workforce', 'fte', 'full-time equivalent', 'employee count',
        'global workforce', 'employee population', 'employee profile'
    ],
    'turnover': [
        'employee turnover', 'attrition rate', 'voluntary turnover',
        'retention rate', 'turnover rate', 'employee retention',
    ],

    # SOCIAL - Development / Training
    'training': [
        'training hours', 'learning hours', 'development hours',
        'training per employee', 'average training', 'employee development',
        'professional development', 'skills training', 'learning and development',
    ],

    # SOCIAL - Diversity
    'diversity': [
        'diversity', 'women in workforce', 'female employees', 'gender diversity',
        'women in management', 'women in leadership', 'minority', 'inclusion',
        'underrepresented', 'demographic', 'gender representation', 'dei', 'held by women'
    ],

    # SOCIAL - Supply Chain
    'supply_chain': [
        'supplier audit', 'supply chain audit', 'vendor assessment',
        'supplier assessment', 'responsible sourcing', 'supplier code',
        'supply chain responsibility', 'supplier evaluation',
    ],

    # SOCIAL - Community
    'community': [
        'community investment', 'charitable', 'donation', 'philanthropy',
        'volunteer hours', 'giving', 'social investment', 'foundation',
        'community engagement', 'corporate giving',
    ],

    # SOCIAL - Privacy & Data
    'data_privacy': [
        'data breach', 'security incident', 'cyber incident', 'privacy breach',
        'data protection', 'cybersecurity', 'information security',
    ],

    # GOVERNANCE - Board
    'board_governance': [
        'board of directors', 'board composition', 'independent director',
        'board diversity', 'board oversight', 'governance structure',
    ],

    # GOVERNANCE - Ethics
    'ethics': [
        'code of ethics', 'code of conduct', 'anti-corruption', 'anti-bribery',
        'ethics training', 'compliance', 'business ethics', 'ethical conduct',
    ],

    # COMPONENT - Financial
    'financial': [
        'revenue', 'total revenue', 'net revenue', 'annual revenue',
        'fiscal year', 'financial performance', 'economic performance'
    ],
}

TABLE_INDICATOR_KEYWORDS = [
    'performance data', 'esg data', 'sustainability data', 'data summary',
    'metrics', 'kpi', 'key performance', 'gri index', 'sasb index',
    'data table', 'performance summary', 'environmental data',
    'social data', 'sustainability metrics', 'esg metrics', 'tcfd',
    'data appendix', 'esg performance', 'sustainability performance',
]

### CDP Mapping

In [50]:
# =============================================================================
# CDP SECTION MAPPING
# =============================================================================
# Maps CDP questionnaire section IDs to our metric issues.
# CDP section IDs are very stable across years and companies.

CDP_SECTION_TO_ISSUE = {
    # C1 - Governance
    'c1.1': 'board_governance',
    'c1.2': 'board_governance',
    'c1.3': 'board_governance',

    # C4 - Targets
    'c4.1': 'carbon_emissions',
    'c4.2': 'carbon_emissions',

    # C5 - Emissions methodology
    'c5.1': 'carbon_emissions',
    'c5.2': 'carbon_emissions',

    # C6 - Emissions data (CRITICAL)
    'c6.1': 'carbon_emissions',     # Scope 1
    'c6.2': 'carbon_emissions',     # Scope 1 breakdown
    'c6.3': 'carbon_emissions',     # Scope 2
    'c6.4': 'carbon_emissions',     # Scope 2 breakdown
    'c6.5': 'carbon_emissions',     # Scope 3
    'c6.6': 'carbon_emissions',     # Scope 3 categories
    'c6.7': 'carbon_emissions',     # Scope 3 breakdown
    'c6.10': 'carbon_emissions',    # Emissions methodology

    # C7 - Emissions breakdowns
    'c7.1': 'carbon_emissions',
    'c7.2': 'carbon_emissions',
    'c7.3': 'carbon_emissions',
    'c7.5': 'carbon_emissions',
    'c7.6': 'carbon_emissions',
    'c7.9': 'carbon_emissions',

    # C8 - Energy
    'c8.1': 'energy',
    'c8.2': 'energy',              # Energy consumption totals
    'c8.2a': 'energy',             # Energy consumption breakdown
    'c8.2b': 'energy',
    'c8.2c': 'energy',
    'c8.2d': 'energy',
    'c8.2e': 'energy',

    # C9 - Additional metrics
    'c9.1': 'energy',

    # W1-W9 - Water (CDP Water Security)
    'w1.2': 'water',
    'w1.2b': 'water',
    'w1.2d': 'water',
    'w5.1': 'water',
    'w5.1a': 'water',
    'w8.1': 'water',

    # C10 - Verification
    'c10.1': 'carbon_emissions',
    'c10.2': 'carbon_emissions',
}

# Regex pattern to match CDP section IDs on a page
CDP_SECTION_PATTERN = re.compile(
    r'\b(c[0-9]{1,2}\.[0-9]{1,2}[a-z]?|w[0-9]{1,2}\.[0-9]{1,2}[a-z]?)\b',
    re.IGNORECASE
)

print(f"✅ CDP section mapping defined: {len(CDP_SECTION_TO_ISSUE)} sections")

✅ CDP section mapping defined: 37 sections


### Keywords DEF 14A

In [51]:
# =============================================================================
# DEF 14A KEYWORD DEFINITIONS — REGEX-BASED FOR PHRASING VARIATION
# =============================================================================

SEC_DEF14A_PATTERNS = {
    'board': {
        'patterns': [
            r'election\s+of\s+directors?',
            r'nominees?\s+for\s+election',
            r'director\s+nominees?',
            r'board\s+of\s+directors?',
            r'board\s+composition',
            r'board\s+diversity(?:\s+matrix)?',
            r'director\s+(?:independence|qualifications?)',
            r'independent\s+directors?',
            r'non[\-\s]?employee\s+directors?',
            r'continuing\s+directors?',
            r'(?:number|size)\s+of\s+(?:the\s+)?board',
            r'class\s+[iIvV]{1,3}\s+directors?',
        ],
        'weight': 3,
    },
    'diversity': {
        'patterns': [
            r'(?:board|director)\s+diversity',
            r'diversity\s+matrix',
            r'women\s+(?:on|serving\s+on)\s+(?:the\s+)?board',
            r'female\s+directors?',
            r'gender\s+diversity',
            r'self[\-\s]?identif(?:ied|ication)',
            r'(?:african\s+american|hispanic|asian|lgbtq|ethnicit|race\s+and)',
            r'demographic\s+(?:background|data|information|composition)',
        ],
        'weight': 4,
    },
    'meetings': {
        'patterns': [
            r'board\s+meeting',
            r'(?:meeting|director)\s+attendance',
            r'committees?\s+of\s+the\s+board',
            r'number\s+of\s+meetings?',
            r'attendance\s+rate',
            r'attend(?:ed|ing)\s+(?:at\s+least|all|each)',
            r'(?:met|held)\s+\d+\s+times',
        ],
        'weight': 3,
    },
    'pay_ratio': {
        'patterns': [
            r'(?:ceo|chief\s+executive\s+officer)\s+pay\s+ratio',
            r'pay\s+ratio\s+(?:disclosure|information)',
            r'ratio\s+of\s+(?:the\s+)?(?:ceo|chief\s+executive)',
            r'(?:ceo|chief\s+executive)\s+(?:compensation|pay)\s+(?:ratio|rate)',
            r'pay\s+(?:ratio|rate)\b',
            r'median\s+(?:employee|annual\s+total\s+compensation)',
            r'annual\s+total\s+compensation\s+of\s+(?:our\s+)?(?:ceo|chief)',
            r'(?:ratio|multiple)\s+of\s+(?:annual|total)\s+compensation',
            r'\d{1,3}(?:,\d{3})*\s*(?:to|:)\s*1\b',
            r'\d+\s+times\s+(?:the\s+)?median',
        ],
        'weight': 6,
    },
'say_on_pay': {
        'patterns': [
            # Core phrase
            r'say[\-\s]on[\-\s]pay',
            # Vote/advisory general
            r'advisory\s+(?:vote|approval)\s+(?:on|to\s+approve)\s+(?:executive\s+)?compensation',
            r'approv(?:e|al)\s+(?:of\s+)?(?:named\s+)?executive\s+(?:officer\s+)?compensation',
            r'non[\-\s]?binding\s+(?:advisory\s+)?vote',
            # Proposal references
            r'(?:proposal|item)\s+(?:no\.?\s*)?[23]\b.*?(?:compensation|pay|advisory)',
            # Vote results — generic
            r'(?:for|against|abstain)\s+(?:votes?|ballots?)',
            r'(?:percent|%)\s+of\s+votes?\s+cast',
            r'approval\s+(?:rate|percentage)',
            # ── Result-page data patterns (cross-company) ──
            r'received\s+(?:approximately\s+)?\d+\.?\d*\s*%',
            r'\d+\.?\d*\s*%\s*(?:support|approval|of\s+(?:the\s+)?votes)',
            r'\d+\s*%\s+of\s+votes\s+cast\s+on\s+the\s+say',
            r'(?:stockholder|shareholder)\s+support',
            r'(?:say[\-\s]on[\-\s]pay|advisory)\s+(?:vote\s+)?results?',
            r'(?:say[\-\s]on[\-\s]pay|advisory)\s+resolution',
            r'support\s+(?:for|of)\s+(?:the\s+)?(?:compensation|executive)',
            r'response\s+to\s+.*?say[\-\s]on[\-\s]pay',
            r'voted\s+in\s+favor',
            r'consideration\s+of\s+.*?say[\-\s]on[\-\s]pay',
            r'results\s+of\s+(?:the\s+)?say[\-\s]on[\-\s]pay',
            r'votes?\s+cast\s+on\s+(?:the\s+)?(?:say[\-\s]on[\-\s]pay|advisory)',
        ],
        'weight': 5,
    },    'ethics': {
        'patterns': [
            r'code\s+of\s+(?:ethics|conduct|business\s+conduct)',
            r'corporate\s+governance\s+(?:guidelines|principles)',
            r'anti[\-\s]?(?:corruption|bribery)',
            r'(?:compliance|ethics)\s+(?:program|policy|training)',
            r'whistleblower\s+(?:policy|protection|program|hotline)',
            r'standards?\s+of\s+(?:business\s+)?conduct',
        ],
        'weight': 2,
    },
    'ownership': {
        'patterns': [
            r'(?:security|stock|beneficial)\s+ownership',
            r'(?:dual|multi)[\-\s]?class\s+(?:share|stock|structure)',
            r'voting\s+(?:power|rights|structure)',
            r'(?:class\s+[ab]|common\s+stock)\s+(?:shares?|voting)',
            r'(?:shares?|stock)\s+outstanding',
            r'principal\s+(?:stock|share)holders?',
        ],
        'weight': 2,
    },
}

print("✅ DEF 14A regex patterns defined")
print(f"   Categories: {len(SEC_DEF14A_PATTERNS)}")
print(f"   Total patterns: {sum(len(d['patterns']) for d in SEC_DEF14A_PATTERNS.values())}")

✅ DEF 14A regex patterns defined
   Categories: 7
   Total patterns: 69


### Keywords 10K

In [52]:
# =============================================================================
# 10-K KEYWORDS
# =============================================================================

SEC_10K_PATTERNS = {
    'tax': {
        'patterns': [
            r'effective\s+tax\s+rate',
            r'(?:provision|benefit)\s+for\s+income\s+tax',
            r'tax\s+rate\s+reconciliation',
            r'deferred\s+tax\s+(?:assets?|liabilit)',
            r'(?:income|federal|state|foreign)\s+tax\s+(?:expense|benefit|provision)',
            r'statutory\s+(?:tax\s+)?rate',
            r'rate\s+reconciliation',
            r'(?:domestic|foreign)\s+(?:earnings|income)\s+before\s+(?:income\s+)?tax',
            r'tax\s+(?:credits?|holidays?|incentives?)',
            r'unrecognized\s+tax\s+benefits?',
            r'valuation\s+allowance',
            r'effective\s+(?:income\s+)?tax\s+rates?\s+(?:of|was|were)',
            r'effective\s+(?:income\s+)?tax\s+rates?\s+.*?\d+\.?\d*\s*%',
            r'note\s+\d{1,2}\s*[\-–—:.]?\s*(?:income\s+tax|tax|provision)',
            r'income\s+tax\s*(?:expense|benefit)',
        ],
        'weight': 5,
    },
    'accounting': {
        'patterns': [
            r'material\s+weakness',
            r'internal\s+control\s+over\s+financial\s+reporting',
            r'restatement',
            r'significant\s+deficiency',
            r'remediation',
            r'(?:control|accounting)\s+deficiency',
            r'(?:management|auditor)(?:\'?s)?\s+report\s+on\s+internal\s+control',
        ],
        'weight': 3,
    },
    'geographic_revenue': {
        'patterns': [
        # Section headers
        r'geographic\s+(?:data|information|breakdown|summary)',
        r'segment\s+(?:information|reporting|data|results)',
        r'(?:reportable|operating|business)\s+segments?',
        r'geographic\s+(?:regions?|areas?|markets?)',
        # Revenue by geography
        r'(?:revenue|sales|net\s+revenue)\s+by\s+(?:region|geography|country|segment)',
        r'(?:geographic|regional)\s+(?:revenue|sales|income|results)',
        r'(?:domestic|international|foreign)\s+(?:revenue|sales|net\s+revenue|operations)',
        # Country/region names near financial data
        r'(?:united\s+states|americas?|europe|asia|apac|emea)\s+.*?\$?\s*\d',
        # Explicit country-by-country
        r'country[\-\s]by[\-\s]country',
        r'sales\s+to\s+external\s+customers',
        r'(?:revenue|sales|income)\s+(?:from|in)\s+(?:foreign|international|external)',
        # Property/long-lived assets by geography (also common ESG disclosure)
        r'(?:long[\-\s]?lived\s+assets?|property)\s+by\s+(?:geographic|country|region)',
        ],
        'weight': 4,
    },
    'employees': {
        'patterns': [
            r'(?:total|approximately)\s+\d[\d,]*\s+(?:employees|team\s+members|associates)',
            r'(?:we|the\s+company)\s+(?:had|employed|have)\s+(?:approximately\s+)?\d[\d,]*\s+(?:full[\-\s]?time|employees)',
            r'(?:number|headcount)\s+of\s+employees',
            r'human\s+capital\s+(?:resources?|management|disclosures?)',
            r'(?:our|the)\s+(?:global\s+)?workforce',
            r'employee\s+(?:count|headcount|population)',
            r'full[\-\s]?time\s+equivalent',
        ],
        'weight': 4,
    },
}

## Helper Functions

In [53]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def get_exclusion_penalty(text_lower: str, exclusion_keywords: list) -> int:
    """Check if a page matches exclusion patterns. Returns negative penalty."""
    matches = sum(1 for kw in exclusion_keywords if kw in text_lower)
    if matches == 0:
        return 0

    # Strong exclusions — page is almost certainly not data
    strong = [
        #'table of contents',
        'forward-looking statements',
        'cautionary statement', 'safe harbor', 'legal disclaimer',
        'gri content index', 'how to vote', 'voting instructions',
    ]
    if any(kw in text_lower for kw in strong):
        return -100  # Effectively kill this page

    return -15 * matches


def is_data_table(table) -> Tuple[bool, int]:
    """
    Determine if a table is a genuine data table vs TOC/layout table.
    Returns (is_data, quality_score).
    """
    if not table or len(table) < 2:
        return False, 0

    numeric_rows = 0
    total_numeric_cells = 0

    for row in table:
        if not row:
            continue
        row_has_number = False
        for cell in row:
            cell_str = str(cell) if cell else ''
            # Real data: comma-formatted numbers, decimals, percentages
            if re.search(r'\b\d{1,3}(?:,\d{3})+\b|\b\d+\.\d+\b|\d+%', cell_str):
                total_numeric_cells += 1
                row_has_number = True
        if row_has_number:
            numeric_rows += 1

    if numeric_rows < 2:
        return False, 0

    # Check for year headers (strong signal)
    header_str = str(table[0]).lower() if table[0] else ''
    has_year_headers = sum(1 for y in YEAR_KEYWORDS if y in header_str)

    quality = 0
    if numeric_rows >= 3 and total_numeric_cells >= 4:
        quality = 15
    elif numeric_rows >= 2:
        quality = 8

    if has_year_headers >= 2:
        quality += 10
    if any(u in str(table).lower() for u in ['tco2e', 'mwh', '%', 'tonnes', 'hours']):
        quality += 5

    return quality > 0, quality


def count_data_numbers(text: str) -> int:
    """Count meaningful data numbers on a page (not page/footnote numbers)."""
    comma_nums = len(re.findall(r'\b\d{1,3}(?:,\d{3})+\b', text))
    decimals = len(re.findall(r'\b\d+\.\d+\b', text))
    percentages = len(re.findall(r'\b\d+(?:\.\d+)?%', text))
    large_ints = len(re.findall(r'\b\d{3,}\b', text))
    return comma_nums + decimals + percentages + large_ints


def add_adjacent_pages(
    selected: Set[int],
    page_scores: list,
    total_pages: int,
    threshold: float,
) -> List[int]:
    """Add ±1 adjacent pages if they score above threshold."""
    adjacent_added = []
    snapshot = set(selected)
    for page_num in snapshot:
        for adj in [page_num - 1, page_num + 1]:
            if adj < 1 or adj > total_pages or adj in selected:
                continue
            adj_page_scores = [p for p in page_scores if p['page_num'] == adj]
            if adj_page_scores:
                best = max(s['score'] for s in adj_page_scores)
                if best >= threshold:
                    selected.add(adj)
                    adjacent_added.append(adj)
    return adjacent_added

## Filter Functions

### Sustainability Report Filter

In [54]:
# =============================================================================
# FILTER: SUSTAINABILITY — STANDARD REPORTS
# =============================================================================

def _filter_standard_sustainability(pdf_path: str, config: Dict) -> Tuple[List[int], Dict]:
    """Filter a standard sustainability PDF (presentations, annual reports)."""

    log_data = {
        'doc_type': DocType.STANDARD,
        'issue_selections': {},
        'table_indicator_pages': [],
        'adjacent_pages_added': [],
        'excluded_pages': [],
    }

    page_issue_scores = []
    page_table_scores = []
    excluded_pages = set()

    try:
        with pdfplumber.open(pdf_path) as pdf:
            log_data['total_pages'] = len(pdf.pages)

            for i, page in enumerate(pdf.pages):
                if i >= 200:
                    break
                page_num = i + 1

                try:
                    text = page.extract_text() or ''
                    text_lower = text.lower()
                    tables = page.extract_tables()

                    # Exclusion check
                    penalty = get_exclusion_penalty(text_lower, SUSTAINABILITY_EXCLUSION_KEYWORDS)

                    # # Dedicated TOC page detection: a real TOC page has
                    # # "table of contents" AND many lines with dotted leaders / page numbers
                    # if 'table of contents' in text_lower:
                    #     lines = [l.strip() for l in text.split('\n') if l.strip()]
                    #     toc_like_lines = sum(
                    #         1 for l in lines
                    #         if re.search(r'\.{3,}\s*\d+\s*$', l)
                    #         or re.search(r'\b\d{1,3}\s*$', l.strip())
                    #     )
                    #     if toc_like_lines >= 5 or (len(lines) > 0 and toc_like_lines / len(lines) > 0.3):
                    #         excluded_pages.add(page_num)
                    #         continue

                    if penalty <= -100:
                        excluded_pages.add(page_num)
                        continue

                    # Data evidence signals
                    data_count = count_data_numbers(text)
                    best_table_quality = 0
                    if tables:
                        for t in tables:
                            _, q = is_data_table(t)
                            best_table_quality = max(best_table_quality, q)

                    has_data = data_count >= 3 or best_table_quality >= 8

                    # Score for each issue
                    for issue, keywords in SUSTAINABILITY_ISSUE_KEYWORDS.items():
                        matched = [kw for kw in keywords if kw in text_lower]
                        if not matched:
                            continue

                        score = len(matched)

                        if has_data:
                            years_found = sum(1 for y in YEAR_KEYWORDS if y in text)
                            units_found = sum(1 for u in UNIT_KEYWORDS if u in text_lower)
                            score += years_found * 2
                            score += units_found * 3
                            score += best_table_quality
                        else:
                            score = min(score, 3)

                        score += penalty

                        if score > 0:
                            page_issue_scores.append({
                                'page_num': page_num,
                                'issue': issue,
                                'score': score,
                                'keywords': matched[:5],
                                'has_data': has_data,
                            })

                    # Table indicator scoring
                    table_matched = [kw for kw in TABLE_INDICATOR_KEYWORDS if kw in text_lower]
                    if table_matched and has_data:
                        t_score = len(table_matched) * 5 + best_table_quality
                        t_score += sum(1 for y in YEAR_KEYWORDS if y in text) * 3
                        page_table_scores.append({
                            'page_num': page_num,
                            'score': t_score,
                            'keywords': table_matched[:5],
                        })

                except Exception:
                    continue

    except Exception as e:
        log_data['error'] = str(e)
        return [], log_data

    # ── Page selection ──
    top_per = config.get('top_pages_per_issue', 3)
    min_score = config.get('min_issue_score', 2)
    gap_ratio = config.get('score_gap_ratio', 0.25)

    selected = set()
    issue_selections = {}

    for issue in SUSTAINABILITY_ISSUE_KEYWORDS.keys():
        issue_pages = sorted(
            [p for p in page_issue_scores if p['issue'] == issue],
            key=lambda x: -x['score']
        )
        top_score = issue_pages[0]['score'] if issue_pages else 0
        top_pages = []

        for p in issue_pages:
            if len(top_pages) >= top_per:
                break
            if p['score'] < min_score:
                break
            if top_pages and p['score'] < top_score * gap_ratio:
                break
            top_pages.append(p)
            selected.add(p['page_num'])

        issue_selections[issue] = {
            'pages': [p['page_num'] for p in top_pages],
            'scores': [p['score'] for p in top_pages],
            'total_candidates': len(issue_pages),
        }

    # Table indicator pages (non-overlapping only)
    page_table_scores.sort(key=lambda x: -x['score'])
    table_pages = []
    for p in page_table_scores:
        if p['page_num'] not in selected and len(table_pages) < config.get('top_table_indicator_pages', 5):
            selected.add(p['page_num'])
            table_pages.append({'page_num': p['page_num'], 'score': p['score']})

    # Adjacent pages
    adj_threshold = config.get('adjacent_page_threshold', 0.4) * min_score
    adj_added = add_adjacent_pages(selected, page_issue_scores, log_data.get('total_pages', 0), adj_threshold)

    # Apply cap
    selected_pages = sorted(selected)
    max_pages = config.get('max_pages_to_select', 50)
    if len(selected_pages) > max_pages:
        combined = defaultdict(float)
        for p in page_issue_scores:
            if p['page_num'] in selected:
                combined[p['page_num']] += p['score']
        for p in page_table_scores:
            if p['page_num'] in selected:
                combined[p['page_num']] += p['score']
        top = sorted(combined.items(), key=lambda x: -x[1])[:max_pages]
        selected_pages = sorted(p for p, _ in top)

    log_data['selected_pages'] = selected_pages
    log_data['issue_selections'] = issue_selections
    log_data['table_indicator_pages'] = table_pages
    log_data['adjacent_pages_added'] = adj_added
    log_data['excluded_pages'] = sorted(excluded_pages)

    return selected_pages, log_data

### CDP Questionaire Filter

In [55]:
# =============================================================================
# FILTER: SUSTAINABILITY — CDP QUESTIONNAIRES
# =============================================================================

def _filter_cdp_questionnaire(pdf_path: str, config: Dict) -> Tuple[List[int], Dict]:
    """Filter CDP questionnaire using section ID matching."""

    log_data = {
        'doc_type': DocType.CDP,
        'section_matches': {},
        'issue_coverage': {},
        'excluded_pages': [],
    }

    page_sections = {}
    all_issues_found = defaultdict(list)

    try:
        with pdfplumber.open(pdf_path) as pdf:
            log_data['total_pages'] = len(pdf.pages)

            for i, page in enumerate(pdf.pages):
                if i >= 300:
                    break
                page_num = i + 1

                try:
                    text = page.extract_text() or ''
                    text_lower = text.lower()

                    found_sections = CDP_SECTION_PATTERN.findall(text_lower)

                    if found_sections:
                        page_matches = []
                        for section_id in found_sections:
                            section_id = section_id.lower()
                            issue = CDP_SECTION_TO_ISSUE.get(section_id)
                            if issue:
                                page_matches.append((section_id, issue))
                                all_issues_found[issue].append(page_num)
                        if page_matches:
                            page_sections[page_num] = page_matches

                    # Keyword fallback for non-standard CDP formatting
                    for issue, keywords in SUSTAINABILITY_ISSUE_KEYWORDS.items():
                        matched = [kw for kw in keywords if kw in text_lower]
                        if len(matched) >= 3:
                            data_count = count_data_numbers(text)
                            if data_count >= 2:
                                if page_num not in page_sections:
                                    page_sections[page_num] = []
                                page_sections[page_num].append(('keyword_match', issue))
                                all_issues_found[issue].append(page_num)

                except Exception:
                    continue

    except Exception as e:
        log_data['error'] = str(e)
        return [], log_data

    selected = set()
    context_pages = config.get('cdp_context_pages', 1)

    for page_num in page_sections:
        selected.add(page_num)
        for delta in range(1, context_pages + 1):
            if page_num + delta <= log_data.get('total_pages', 999):
                selected.add(page_num + delta)

    issue_coverage = {}
    for issue, pages in all_issues_found.items():
        issue_coverage[issue] = sorted(set(pages))

    selected_pages = sorted(selected)

    max_pages = config.get('max_pages_to_select', 50)
    if len(selected_pages) > max_pages:
        page_match_count = {pn: len(secs) for pn, secs in page_sections.items()}
        direct_pages = sorted(page_sections.keys())
        if len(direct_pages) <= max_pages:
            selected_pages = direct_pages
        else:
            top = sorted(page_match_count.items(), key=lambda x: -x[1])[:max_pages]
            selected_pages = sorted(p for p, _ in top)

    log_data['selected_pages'] = selected_pages
    log_data['section_matches'] = {
        pn: [(s, i) for s, i in secs]
        for pn, secs in page_sections.items()
    }
    log_data['issue_coverage'] = issue_coverage

    return selected_pages, log_data

### Website-To-PDF Document Filter

In [56]:
# =============================================================================
# FILTER: SUSTAINABILITY — WEBSITE-TO-PDF
# =============================================================================

def _filter_web_pdf(pdf_path: str, config: Dict) -> Tuple[List[int], Dict]:
    """Filter website-to-PDF sustainability documents."""

    log_data = {
        'doc_type': DocType.WEB_PDF,
        'issue_selections': {},
        'excluded_pages': [],
    }

    web_exclusions = SUSTAINABILITY_EXCLUSION_KEYWORDS + [
        'cookie policy', 'privacy policy', 'terms of use',
        'subscribe to our newsletter', 'follow us on',
        'share this page', 'skip to main content',
        'all rights reserved',
    ]

    page_issue_scores = []
    excluded_pages = set()

    try:
        with pdfplumber.open(pdf_path) as pdf:
            log_data['total_pages'] = len(pdf.pages)

            for i, page in enumerate(pdf.pages):
                if i >= 200:
                    break
                page_num = i + 1

                try:
                    text = page.extract_text() or ''
                    text_lower = text.lower()

                    penalty = get_exclusion_penalty(text_lower, web_exclusions)
                    if penalty <= -100:
                        excluded_pages.add(page_num)
                        continue

                    data_count = count_data_numbers(text)

                    for issue, keywords in SUSTAINABILITY_ISSUE_KEYWORDS.items():
                        matched = [kw for kw in keywords if kw in text_lower]
                        if not matched:
                            continue

                        min_kw = config.get('web_min_keywords_per_page', 1)
                        if len(matched) < min_kw:
                            continue

                        score = len(matched)

                        if data_count >= 5:
                            score += data_count
                            years = sum(1 for y in YEAR_KEYWORDS if y in text)
                            score += years * 3
                            units = sum(1 for u in UNIT_KEYWORDS if u in text_lower)
                            score += units * 4
                        elif data_count >= 2:
                            score += data_count

                        score += penalty

                        if score > 0:
                            page_issue_scores.append({
                                'page_num': page_num,
                                'issue': issue,
                                'score': score,
                                'keywords': matched[:5],
                                'data_count': data_count,
                            })

                except Exception:
                    continue

    except Exception as e:
        log_data['error'] = str(e)
        return [], log_data

    top_per = config.get('web_top_pages_per_issue', 4)
    min_score = 1
    gap_ratio = config.get('score_gap_ratio', 0.25)

    selected = set()
    issue_selections = {}

    for issue in SUSTAINABILITY_ISSUE_KEYWORDS.keys():
        issue_pages = sorted(
            [p for p in page_issue_scores if p['issue'] == issue],
            key=lambda x: -x['score']
        )
        top_score = issue_pages[0]['score'] if issue_pages else 0
        top_pages = []

        for p in issue_pages:
            if len(top_pages) >= top_per:
                break
            if p['score'] < min_score:
                break
            if top_pages and p['score'] < top_score * gap_ratio:
                break
            top_pages.append(p)
            selected.add(p['page_num'])

        issue_selections[issue] = {
            'pages': [p['page_num'] for p in top_pages],
            'scores': [p['score'] for p in top_pages],
            'total_candidates': len(issue_pages),
        }

    adj_added = add_adjacent_pages(selected, page_issue_scores, log_data.get('total_pages', 0), 0.5)

    selected_pages = sorted(selected)
    max_pages = config.get('max_pages_to_select', 50)
    if len(selected_pages) > max_pages:
        combined = defaultdict(float)
        for p in page_issue_scores:
            if p['page_num'] in selected:
                combined[p['page_num']] += p['score']
        top = sorted(combined.items(), key=lambda x: -x[1])[:max_pages]
        selected_pages = sorted(p for p, _ in top)

    log_data['selected_pages'] = selected_pages
    log_data['issue_selections'] = issue_selections
    log_data['adjacent_pages_added'] = adj_added
    log_data['excluded_pages'] = sorted(excluded_pages)

    return selected_pages, log_data

## Main Sustainability Filter

In [57]:
# =============================================================================
# MAIN SUSTAINABILITY FILTER — ROUTES TO SUB-FILTER
# =============================================================================

def filter_sustainability_pdf(pdf_path: str, config: Dict = None) -> Tuple[List[int], Dict]:
    """
    Filter sustainability PDF by detecting document type and routing
    to the appropriate sub-filter.
    """
    if config is None:
        config = SUSTAINABILITY_CONFIG

    pdf_path = str(pdf_path)
    filename = Path(pdf_path).name

    doc_type = detect_sustainability_doc_type(pdf_path, filename)
    print(f"         📋 Detected doc type: {doc_type}")

    if doc_type == DocType.CDP:
        selected, log_data = _filter_cdp_questionnaire(pdf_path, config)
    elif doc_type == DocType.WEB_PDF:
        selected, log_data = _filter_web_pdf(pdf_path, config)
    else:
        selected, log_data = _filter_standard_sustainability(pdf_path, config)

    log_data['filename'] = filename
    if 'total_pages' not in log_data:
        log_data['total_pages'] = 0
    log_data['filtering_config'] = config

    return selected, log_data


print("✅ Sustainability PDF filtering defined (3 strategies: standard, CDP, web)")

✅ Sustainability PDF filtering defined (3 strategies: standard, CDP, web)


## Main DEF 14A Filter

In [58]:
# =============================================================================
# CELL: DEF 14A PDF FILTERING — V10 REGEX-BASED
# =============================================================================

def filter_def14a_pdf(pdf_path: str, config: Dict = None) -> Tuple[List[int], Dict]:
    """
    Filter DEF 14A PDF using regex patterns for phrasing variation.
    Catches: "CEO Pay Rate", "Pay Ratio", "Compensation Ratio",
    "Director Independence", footnotes, varied proposal numbering, etc.
    """
    if config is None:
        config = DEF14A_CONFIG

    pdf_path = str(pdf_path)
    filename = Path(pdf_path).name

    log_data = {
        'filename': filename,
        'total_pages': 0,
        'selected_pages': [],
        'page_matches': [],
        'category_selections': {},
        'adjacent_pages_added': [],
        'excluded_pages': [],
        'filtering_config': config,
    }

    try:
        doc = fitz.open(pdf_path)
        log_data['total_pages'] = len(doc)

        page_category_scores = []
        excluded_pages = set()

        # Pre-compile regex patterns
        compiled_patterns = {}
        for category, data in SEC_DEF14A_PATTERNS.items():
            compiled_patterns[category] = {
                'patterns': [re.compile(p, re.IGNORECASE) for p in data['patterns']],
                'weight': data['weight'],
            }

        for page_idx in range(len(doc)):
            page = doc[page_idx]
            text = page.get_text()
            text_lower = text.lower()
            page_num = page_idx + 1

            # Exclusion check
            penalty = get_exclusion_penalty(text_lower, DEF14A_EXCLUSION_KEYWORDS)
            if penalty <= -100:
                excluded_pages.add(page_num)
                continue

            for category, cat_data in compiled_patterns.items():
                matched_patterns = []
                for pattern in cat_data['patterns']:
                    if pattern.search(text):
                        matched_patterns.append(pattern.pattern[:50])

                if not matched_patterns:
                    continue

                # Score: number of distinct patterns matched × weight
                score = len(matched_patterns) * cat_data['weight']
                score += penalty

                if score >= config.get('min_category_score', 1):
                    page_category_scores.append({
                        'page_num': page_num,
                        'category': category,
                        'score': score,
                        'matched_patterns': matched_patterns,
                        'keywords': matched_patterns,  # For compatibility with test functions
                    })

        doc.close()

        # Select top N pages per category
        top_per = config.get('top_pages_per_category', 3)
        min_score = config.get('min_category_score', 1)

        selected = set()
        category_selections = {}

        for category in SEC_DEF14A_PATTERNS.keys():
            cat_pages = sorted(
                [p for p in page_category_scores if p['category'] == category],
                key=lambda x: -x['score']
            )

            top_pages = []
            for p in cat_pages:
                if p['score'] >= min_score and len(top_pages) < top_per:
                    top_pages.append(p)
                    selected.add(p['page_num'])

            category_selections[category] = {
                'pages': [p['page_num'] for p in top_pages],
                'scores': [p['score'] for p in top_pages],
                'matched_patterns': [p['matched_patterns'][:3] for p in top_pages],
                'total_candidates': len(cat_pages),
            }

        # Adjacent pages
        adj_threshold = config.get('adjacent_page_threshold', 0.4) * min_score
        adj_added = add_adjacent_pages(
            selected, page_category_scores, log_data['total_pages'], adj_threshold
        )

        selected_pages = sorted(selected)

        # Apply cap
        max_pages = config.get('max_pages_to_select', 30)
        if len(selected_pages) > max_pages:
            combined = defaultdict(float)
            for p in page_category_scores:
                if p['page_num'] in selected:
                    combined[p['page_num']] += p['score']
            top = sorted(combined.items(), key=lambda x: -x[1])[:max_pages]
            selected_pages = sorted(p for p, _ in top)

        # Build page_matches for logging
        page_total_scores = defaultdict(float)
        page_categories = defaultdict(list)
        for p in page_category_scores:
            pn = p['page_num']
            page_total_scores[pn] += p['score']
            page_categories[pn].append(f"{p['category']}({p['score']:.0f})")

        page_matches = [
            {'page_num': pn, 'score': page_total_scores[pn], 'categories': page_categories[pn]}
            for pn in sorted(page_total_scores.keys())
        ]

        log_data['selected_pages'] = selected_pages
        log_data['page_matches'] = page_matches
        log_data['category_selections'] = category_selections
        log_data['adjacent_pages_added'] = adj_added
        log_data['excluded_pages'] = sorted(excluded_pages)

        return selected_pages, log_data

    except Exception as e:
        log_data['error'] = str(e)
        return [], log_data


print("✅ DEF 14A PDF filtering defined (regex-based)")

✅ DEF 14A PDF filtering defined (regex-based)


In [59]:
# =============================================================================
# OPTIONAL: API-ASSISTED DEF 14A CLASSIFICATION
# =============================================================================

def filter_def14a_with_api(
    pdf_path: str,
    config: Dict = None,
    api_key: str = None,
) -> Tuple[List[int], Dict]:
    """
    Two-stage DEF 14A filter:
    1. Broad regex pre-filter (keeps ~30 candidate pages)
    2. Send candidate page texts to Haiku for precise classification

    Cost: ~$0.007 per document with Haiku.

    Set config['use_api_classification'] = True and provide api_key.
    Falls back to regex-only if API call fails.
    """
    import anthropic

    if config is None:
        config = DEF14A_CONFIG

    # Stage 1: Regex pre-filter (use very low threshold to be inclusive)
    broad_config = {**config, 'min_category_score': 1, 'top_pages_per_category': 8, 'max_pages_to_select': 40}
    candidate_pages, log_data = filter_def14a_pdf(pdf_path, broad_config)

    if not config.get('use_api_classification') or not api_key:
        return candidate_pages, log_data

    # Stage 2: API classification
    try:
        doc = fitz.open(pdf_path)

        # Build page texts for candidates
        page_texts = {}
        for page_num in candidate_pages:
            page = doc[page_num - 1]
            text = page.get_text()[:1500]  # Truncate to save tokens
            page_texts[page_num] = text

        doc.close()

        # Build the classification prompt
        categories_desc = (
            "board composition/size/independence, board diversity/demographics, "
            "board meeting attendance, CEO pay ratio, say-on-pay vote results, "
            "ethics/code of conduct, ownership structure/dual class shares"
        )

        pages_block = "\n\n".join(
            f"=== PAGE {pn} ===\n{text}"
            for pn, text in page_texts.items()
        )

        client = anthropic.Anthropic(api_key=api_key)
        response = client.messages.create(
            model=config.get('api_model', 'claude-haiku-4-5-20251001'),
            max_tokens=500,
            messages=[{
                "role": "user",
                "content": f"""You are classifying pages from a DEF 14A proxy filing.
For each page, determine if it contains ACTUAL DATA (numbers, tables, specific values) for any of these categories: {categories_desc}

Pages that only mention these topics in passing or in a table of contents do NOT count.

Return ONLY a JSON object mapping page numbers to their categories. Example:
{{"20": ["board"], "75": ["pay_ratio"], "80": ["say_on_pay"]}}

If a page has no actual data, omit it.

{pages_block}"""
            }],
        )

        # Parse API response
        response_text = response.content[0].text.strip()
        # Clean potential markdown wrapping
        response_text = re.sub(r'^```json\s*', '', response_text)
        response_text = re.sub(r'\s*```$', '', response_text)
        api_pages = json.loads(response_text)

        # Use API results to refine selection
        api_selected = set()
        for page_str, categories in api_pages.items():
            api_selected.add(int(page_str))

        # Always include pages from both regex and API (union for safety)
        # But if API returned results, prioritize its pages
        if api_selected:
            # Take union of regex top-3-per-category + all API-confirmed pages
            regex_config = {**config, 'top_pages_per_category': 2}
            regex_pages, _ = filter_def14a_pdf(pdf_path, regex_config)
            final_pages = sorted(set(regex_pages) | api_selected)

            log_data['api_classification'] = api_pages
            log_data['api_selected_pages'] = sorted(api_selected)
            log_data['selected_pages'] = final_pages
            return final_pages, log_data

    except Exception as e:
        log_data['api_error'] = str(e)
        # Fall back to regex-only results
        pass

    return candidate_pages, log_data

## Main 10K FIlter

In [60]:
# =============================================================================
# 10-K PDF FILTER
# =============================================================================

def filter_10k_pdf(pdf_path: str, config: Dict = None) -> Tuple[List[int], Dict]:
    """
    Filter 10-K PDF filing for ESG-relevant pages.

    Uses regex patterns to find pages with tax data, employee disclosures,
    geographic revenue, human capital, environmental liabilities, climate risk,
    cybersecurity, and supply chain information.
    """
    if config is None:
        config = TENK_CONFIG

    pdf_path = str(pdf_path)
    filename = Path(pdf_path).name

    log_data = {
        'filename': filename,
        'total_pages': 0,
        'selected_pages': [],
        'page_matches': [],
        'group_selections': {},
        'adjacent_pages_added': [],
        'filtering_config': config,
    }

    try:
        doc = fitz.open(pdf_path)
        log_data['total_pages'] = len(doc)

        page_group_scores = []

        # Pre-compile patterns
        compiled_patterns = {}
        for group, data in SEC_10K_PATTERNS.items():
            compiled_patterns[group] = {
                'patterns': [re.compile(p, re.IGNORECASE) for p in data['patterns']],
                'weight': data['weight'],
            }

        for page_idx in range(len(doc)):
            page = doc[page_idx]
            text = page.get_text()
            page_num = page_idx + 1

            # Skip very short pages (cover pages, blank pages)
            if len(text.strip()) < 100:
                continue

            for group, group_data in compiled_patterns.items():
                matched_patterns = []
                for pattern in group_data['patterns']:
                    if pattern.search(text):
                        matched_patterns.append(pattern.pattern[:60])

                if not matched_patterns:
                    continue

                score = len(matched_patterns) * group_data['weight']

                # Bonus for numeric data (tables with actual values)
                data_count = count_data_numbers(text)
                if data_count >= 10:
                    score += 10
                elif data_count >= 5:
                    score += 5

                # Bonus for percentage patterns (tax rates, etc.)
                pct_count = len(re.findall(r'\d+\.?\d*\s*%', text))
                if pct_count >= 3:
                    score += 5

                if score >= config.get('min_group_score', 1):
                    page_group_scores.append({
                        'page_num': page_num,
                        'group': group,
                        'score': score,
                        'matched_patterns': matched_patterns,
                        'keywords': matched_patterns,  # Compatibility
                    })

        doc.close()

        # ── Select top N pages per group ──
        top_per = config.get('top_pages_per_group', 3)
        min_score = config.get('min_group_score', 1)
        gap_ratio = config.get('score_gap_ratio', 0.25)

        selected = set()
        group_selections = {}

        for group in SEC_10K_PATTERNS.keys():
            group_pages = sorted(
                [p for p in page_group_scores if p['group'] == group],
                key=lambda x: -x['score']
            )
            top_score = group_pages[0]['score'] if group_pages else 0
            top_pages = []

            for p in group_pages:
                if len(top_pages) >= top_per:
                    break
                if p['score'] < min_score:
                    break
                if top_pages and p['score'] < top_score * gap_ratio:
                    break
                top_pages.append(p)
                selected.add(p['page_num'])

            group_selections[group] = {
                'pages': [p['page_num'] for p in top_pages],
                'scores': [p['score'] for p in top_pages],
                'matched_patterns': [p['matched_patterns'][:3] for p in top_pages],
                'total_candidates': len(group_pages),
            }

        # ── Adjacent pages ──
        adj_threshold = config.get('adjacent_page_threshold', 0.4) * min_score
        adj_added = add_adjacent_pages(
            selected, page_group_scores, log_data['total_pages'], adj_threshold
        )

        selected_pages = sorted(selected)

        # ── Apply cap ──
        max_pages = config.get('max_pages_to_select', 30)
        if len(selected_pages) > max_pages:
            combined = defaultdict(float)
            for p in page_group_scores:
                if p['page_num'] in selected:
                    combined[p['page_num']] += p['score']
            top = sorted(combined.items(), key=lambda x: -x[1])[:max_pages]
            selected_pages = sorted(p for p, _ in top)

        # ── Build page_matches for logging ──
        page_total_scores = defaultdict(float)
        page_groups = defaultdict(list)
        for p in page_group_scores:
            pn = p['page_num']
            page_total_scores[pn] += p['score']
            page_groups[pn].append(f"{p['group']}({p['score']:.0f})")

        page_matches = [
            {'page_num': pn, 'score': page_total_scores[pn], 'groups': page_groups[pn]}
            for pn in sorted(page_total_scores.keys())
        ]

        log_data['selected_pages'] = selected_pages
        log_data['page_matches'] = page_matches
        log_data['group_selections'] = group_selections
        log_data['adjacent_pages_added'] = adj_added

        return selected_pages, log_data

    except Exception as e:
        log_data['error'] = str(e)
        return [], log_data


print("✅ 10-K PDF filtering defined")
print(f"   Keyword groups: {len(SEC_10K_PATTERNS)}")
print(f"   Total patterns: {sum(len(d['patterns']) for d in SEC_10K_PATTERNS.values())}")

✅ 10-K PDF filtering defined
   Keyword groups: 4
   Total patterns: 41


## Renaiming Helper

In [61]:
# =============================================================================
# RENAMING HELPER — for CDP and web PDFs
# =============================================================================

def suggest_renames(company_dir: Path) -> Dict[str, str]:
    """
    Scan a company's documents and suggest renames for CDP/web PDFs.
    Returns dict of {current_name: suggested_name}.

    Run this once to identify which files need renaming.
    """
    suggestions = {}

    for pdf_path in company_dir.glob("*.pdf"):
        if any(x in pdf_path.name for x in ['10-K', '10K', 'DEF_14A', 'DEF14A']):
            continue  # Skip non-sustainability docs

        try:
            with pdfplumber.open(str(pdf_path)) as pdf:
                first_pages = ""
                for page in pdf.pages[:3]:
                    first_pages += (page.extract_text() or "").lower() + "\n"

            doc_type = "standard"

            # CDP detection
            cdp_signals = ['cdp', 'c0.1', 'c1.1', 'climate change 20']
            if sum(1 for s in cdp_signals if s in first_pages) >= 2:
                doc_type = "cdp"

            # Web PDF detection
            web_signals = ['cookie', 'privacy policy', 'skip to content', 'subscribe']
            if sum(1 for s in web_signals if s in first_pages) >= 2:
                doc_type = "web"

            if doc_type != "standard":
                new_name = pdf_path.name.replace("Report_", f"Report_{doc_type.upper()}_")
                suggestions[pdf_path.name] = new_name
                print(f"  {doc_type.upper()}: {pdf_path.name} → {new_name}")

        except Exception:
            continue

    return suggestions

## Export Function

In [ ]:
# =============================================================================
# DOCUMENT EXPORT FUNCTIONS
# =============================================================================

def export_filtered_pdf(source_path: str, selected_pages: List[int],
                        output_path: str, image_config: Dict = IMAGE_FALLBACK_CONFIG) -> Dict:
    """
    Export selected pages to a new filtered PDF.
    Handles bloated PDFs by rendering as images if needed.

    Returns:
    - export_log: Dict with export details
    """
    export_log = {
        'source': str(source_path),
        'output': str(output_path),
        'pages_selected': len(selected_pages),
        'used_image_fallback': False
    }

    try:
        doc = fitz.open(source_path)
        new_doc = fitz.open()

        # First try: normal extraction
        for page_num in selected_pages:
            if 1 <= page_num <= len(doc):
                new_doc.insert_pdf(doc, from_page=page_num-1, to_page=page_num-1)

        pdf_bytes = new_doc.tobytes(garbage=4, deflate=True)
        pdf_mb = len(pdf_bytes) / (1024 * 1024)

        # Check if size is acceptable
        if pdf_mb > image_config['max_pdf_size_mb']:
            # Fallback: render as images
            new_doc.close()
            new_doc = fitz.open()

            for page_num in selected_pages:
                if 1 <= page_num <= len(doc):
                    page = doc[page_num - 1]
                    scale = image_config['target_width'] / page.rect.width
                    pix = page.get_pixmap(matrix=fitz.Matrix(scale, scale))
                    img_bytes = pix.tobytes("jpeg", jpg_quality=image_config['jpeg_quality'])

                    new_page = new_doc.new_page(width=pix.width, height=pix.height)
                    new_page.insert_image(new_page.rect, stream=img_bytes)

            pdf_bytes = new_doc.tobytes(garbage=4, deflate=True)
            export_log['used_image_fallback'] = True

        # Save to output
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        with open(output_path, 'wb') as f:
            f.write(pdf_bytes)

        export_log['output_size_mb'] = len(pdf_bytes) / (1024 * 1024)
        export_log['success'] = True

        new_doc.close()
        doc.close()

        return export_log

    except Exception as e:
        export_log['error'] = str(e)
        export_log['success'] = False
        return export_log


def export_filtered_text_as_pdf(text: str, output_path: str) -> Dict:
    """
    Export filtered text as a simple PDF.

    Returns:
    - export_log: Dict with export details
    """
    export_log = {
        'output': str(output_path),
        'text_length': len(text)
    }

    try:
        doc = fitz.open()

        # Split text into pages
        chunk_size = 3000
        y = 50
        page = doc.new_page(width=612, height=792)

        for i in range(0, len(text), chunk_size):
            chunk = text[i:i+chunk_size]

            if y > 700:
                page = doc.new_page(width=612, height=792)
                y = 50

            rect = fitz.Rect(40, y, 572, 750)
            page.insert_textbox(rect, chunk, fontsize=8)
            y = 750

        # Save
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        doc.save(output_path, garbage=4, deflate=True)

        export_log['pages_created'] = len(doc)
        export_log['success'] = True

        doc.close()

        return export_log

    except Exception as e:
        export_log['error'] = str(e)
        export_log['success'] = False
        return export_log

print("✅ Export functions defined")

✅ Export functions defined


## Main Filtering Pipeline

In [ ]:
# =============================================================================
# MAIN FILTERING PIPELINE
# =============================================================================

def get_year(filename: str) -> Optional[int]:
    """Extract year from filename."""
    for year in YEARS:
        if str(year) in filename:
            return year
    return None


def filter_company_documents(company: str, input_dir: Path, output_dir: Path,
                              log_dir: Path) -> Dict:
    """
    Filter all documents for a single company.

    Returns:
    - company_log: Complete filtering log for the company
    """
    print(f"\n{'='*70}")
    print(f"📁 FILTERING: {company}")
    print(f"{'='*70}")

    company_input = input_dir / company
    company_output = output_dir / company
    company_output.mkdir(parents=True, exist_ok=True)

    company_log = {
        'company': company,
        'timestamp': datetime.now().isoformat(),
        'sustainability_pdfs': [],
        'def14a_pdfs': [],
        'tenk_pdfs': [],
        'summary': {
            'total_files_processed': 0,
            'total_pages_original': 0,
            'total_pages_filtered': 0
        }
    }

    if not company_input.exists():
        print(f"   ⚠️ Input folder not found: {company_input}")
        company_log['error'] = 'Input folder not found'
        return company_log

    # -------------------------------------------------------------------------
    # 1. Filter Sustainability PDFs
    # -------------------------------------------------------------------------
    sustainability_pdfs = [f for f in company_input.glob("*.pdf")
                          if get_year(f.name) in YEARS
                          and not any(x in f.name for x in ['10-K', '10K', 'DEF_14A', 'DEF14A'])]

    print(f"\n   📄 Sustainability PDFs: {len(sustainability_pdfs)}")

    for pdf_path in sorted(sustainability_pdfs):
        filename = pdf_path.name
        print(f"      Processing: {filename}")

        # Filter pages
        selected_pages, filter_log = filter_sustainability_pdf(pdf_path)

        if selected_pages:
            # Export filtered PDF
            output_filename = f"FILTERED_{filename}"
            output_path = company_output / output_filename

            export_log = export_filtered_pdf(pdf_path, selected_pages, output_path)

            print(f"         {filter_log['total_pages']} → {len(selected_pages)} pages", end="")
            if export_log.get('used_image_fallback'):
                print(f" (📸 image fallback)", end="")
            print(f" → {export_log.get('output_size_mb', 0):.2f} MB")

            # Update logs
            filter_log['export'] = export_log
            company_log['sustainability_pdfs'].append(filter_log)
            company_log['summary']['total_pages_original'] += filter_log['total_pages']
            company_log['summary']['total_pages_filtered'] += len(selected_pages)
        else:
            print(f"         ⚠️ No relevant pages found")
            company_log['sustainability_pdfs'].append(filter_log)

    # -------------------------------------------------------------------------
    # 2. Filter DEF 14A PDFs
    # -------------------------------------------------------------------------
    def14a_pdfs = list(company_input.glob("*DEF_14A*.pdf")) + list(company_input.glob("*DEF14A*.pdf"))

    print(f"\n   📋 DEF 14A PDFs: {len(def14a_pdfs)}")

    for pdf_path in sorted(def14a_pdfs):
        filename = pdf_path.name
        print(f"      Processing: {filename}")

        # Filter pages
        selected_pages, filter_log = filter_def14a_pdf(pdf_path)

        if selected_pages:
            # Export filtered PDF
            output_filename = f"FILTERED_{filename}"
            output_path = company_output / output_filename

            export_log = export_filtered_pdf(pdf_path, selected_pages, output_path)

            print(f"         {filter_log['total_pages']} → {len(selected_pages)} pages", end="")
            if export_log.get('used_image_fallback'):
                print(f" (📸 image fallback)", end="")
            print(f" → {export_log.get('output_size_mb', 0):.2f} MB")

            filter_log['export'] = export_log
            company_log['def14a_pdfs'].append(filter_log)
            company_log['summary']['total_pages_original'] += filter_log['total_pages']
            company_log['summary']['total_pages_filtered'] += len(selected_pages)
        else:
            print(f"         ⚠️ No relevant pages found")
            company_log['def14a_pdfs'].append(filter_log)

# -------------------------------------------------------------------------
    # 3. Filter 10-K PDFs
    # -------------------------------------------------------------------------
    tenk_pdfs = list(company_input.glob("*10-K*.pdf")) + list(company_input.glob("*10K*.pdf"))

    print(f"\n   📋 10-K PDFs: {len(tenk_pdfs)}")

    for pdf_path in sorted(tenk_pdfs):
        filename = pdf_path.name
        print(f"      Processing: {filename}")

        # Filter pages
        selected_pages, filter_log = filter_10k_pdf(str(pdf_path))

        if selected_pages:
            # Export filtered PDF
            output_filename = f"FILTERED_{filename}"
            output_path = company_output / output_filename

            export_log = export_filtered_pdf(pdf_path, selected_pages, output_path)

            print(f"         {filter_log['total_pages']} → {len(selected_pages)} pages", end="")
            if export_log.get('used_image_fallback'):
                print(f" (📸 image fallback)", end="")
            print(f" → {export_log.get('output_size_mb', 0):.2f} MB")

            filter_log['export'] = export_log
            company_log['tenk_pdfs'].append(filter_log)
            company_log['summary']['total_pages_original'] += filter_log['total_pages']
            company_log['summary']['total_pages_filtered'] += len(selected_pages)
        else:
            print(f"         ⚠️ No relevant pages found")
            company_log['tenk_pdfs'].append(filter_log)
    # -------------------------------------------------------------------------
    # Summary
    # -------------------------------------------------------------------------
    company_log['summary']['total_files_processed'] = (
        len(sustainability_pdfs) + len(def14a_pdfs) + len(tenk_pdfs)
    )

    reduction_pct = 0
    if company_log['summary']['total_pages_original'] > 0:
        reduction_pct = (1 - company_log['summary']['total_pages_filtered'] /
                         company_log['summary']['total_pages_original']) * 100

    # Store config parameters used for this run
    company_log['summary']['config'] = {
        'sustainability': SUSTAINABILITY_CONFIG,
        'def14a': DEF14A_CONFIG,
        'tenk': TENK_CONFIG,
    }
    company_log['summary']['reduction_pct'] = round(reduction_pct, 1)

    print(f"\n   ✅ Complete:")
    print(f"      Files: {company_log['summary']['total_files_processed']}")
    print(f"      Pages: {company_log['summary']['total_pages_original']} → {company_log['summary']['total_pages_filtered']} ({reduction_pct:.0f}% reduction)")

    # Save log
    log_path = log_dir / f"{company}_filtering_log.json"
    with open(log_path, 'w') as f:
        json.dump(company_log, f, indent=2, default=str)
    print(f"      Log: {log_path}")

    return company_log

print("✅ Main filtering pipeline defined")

✅ Main filtering pipeline defined


## \# Running Pipeline


In [ ]:
# =============================================================================
# RUN FILTERING
# =============================================================================

print(f"{'#'*70}")
print(f"# DOCUMENT FILTERING PIPELINE")
print(f"{'#'*70}")
print(f"Companies to process: {TEST_COMPANIES}")
print(f"Output: {OUTPUT_DIR}")
print(f"{'#'*70}")

######################################################################
# DOCUMENT FILTERING PIPELINE
######################################################################
Companies to process: ['Dell_Technologies', 'HP_Inc']
Output: /content/drive/MyDrive/ESG Data/filtered_documents_4
######################################################################


In [ ]:
# =============================================================================
# PROCESS COMPANIES
# =============================================================================

all_logs = []

for company in TEST_COMPANIES:
    company_log = filter_company_documents(
        company=company,
        input_dir=INPUT_DIR,
        output_dir=OUTPUT_DIR,
        log_dir=LOG_DIR
    )
    all_logs.append(company_log)


📁 FILTERING: Dell_Technologies

   📄 Sustainability PDFs: 7
      Processing: Report_Dell_Technologies_2020_5.pdf
         📋 Detected doc type: standard
         69 → 32 pages → 1.34 MB
      Processing: Report_Dell_Technologies_2021_7.pdf
         📋 Detected doc type: standard
         90 → 31 pages → 7.50 MB
      Processing: Report_Dell_Technologies_2021_8.pdf
         📋 Detected doc type: standard
skipping bad link / annot item 5.
skipping bad link / annot item 6.
         102 → 45 pages → 3.10 MB
      Processing: Report_Dell_Technologies_2022_91.pdf
         📋 Detected doc type: standard
         159 → 44 pages → 13.39 MB
      Processing: Report_Dell_Technologies_2023_15.pdf
         📋 Detected doc type: standard
         381 → 36 pages → 0.08 MB
      Processing: Report_Dell_Technologies_2023_90.pdf
         📋 Detected doc type: standard
         156 → 36 pages → 3.98 MB
      Processing: Report_Dell_Technologies_2024_3.pdf
         📋 Detected doc type: standard
         116 →

In [ ]:
# =============================================================================
# SUMMARY
# =============================================================================

print(f"\n{'='*70}")
print(f"FILTERING COMPLETE")
print(f"{'='*70}")

total_original = sum(log['summary']['total_pages_original'] for log in all_logs)
total_filtered = sum(log['summary']['total_pages_filtered'] for log in all_logs)
total_files = sum(log['summary']['total_files_processed'] for log in all_logs)

print(f"Companies processed: {len(all_logs)}")
print(f"Total files: {total_files}")
print(f"Total pages: {total_original} → {total_filtered}")
if total_original > 0:
    print(f"Reduction: {(1 - total_filtered/total_original)*100:.0f}%")

print(f"\nFiltered documents saved to: {OUTPUT_DIR}")
print(f"Filtering logs saved to: {LOG_DIR}")
print(f"\n→ Now run Pipeline 2 (extraction) on the filtered documents")



FILTERING COMPLETE
Companies processed: 2
Total files: 35
Total pages: 4972 → 1014
Reduction: 80%

Filtered documents saved to: /content/drive/MyDrive/ESG Data/filtered_documents_4
Filtering logs saved to: /content/drive/MyDrive/ESG Data/filtering_logs

→ Now run Pipeline 2 (extraction) on the filtered documents


## \# Run Config Tests

| Test | Folder Suffix | Label | Changes from Baseline |
|------|--------------|-------|----------------------|
| 1 | `_1` | Baseline | V10 initial parameters |
| 2 | `_2` | TOC Removed | Removed 'table of contents' from exclusion keywords |
| 3 | `_3` | top_per_issue=2 | `top_pages_per_issue`: 3→2 |
| 4 | `_4` | +table_ind=3 | Test 3 + `top_table_indicator_pages`: 5→3 |
| 5 | `_5` | +gap=0.5 | Test 4 + `score_gap_ratio`: 0.25→0.5 |
| 6 | `_6` | +no_adjacent | Test 5 + `adjacent_page_threshold`: 0.4→0 (disabled) |
| 7 | `_7` | +def14a=2 | Test 6 + DEF14A `top_pages_per_category`: 3→2 |
| 8 | `_8` | +max=35 | Test 7 + `max_pages_to_select`: 50→35 |


In [ ]:
# =============================================================================
# CELL: AUTOMATED PARAMETER TEST RUNS
# =============================================================================

TEST_COMPANIES = [
    "Advanced_Micro_Devices",
    "Nvidia",
]

# ── Define test configurations ──
# Each test is: (suffix, label, config_overrides)
# Config overrides are applied on top of the DEFAULT configs

DEFAULT_SUSTAINABILITY = {
    "top_pages_per_issue": 3,
    "min_issue_score": 2,
    "top_table_indicator_pages": 5,
    "max_pages_to_select": 50,
    "score_gap_ratio": 0.25,
    "adjacent_page_threshold": 0.4,
    "cdp_context_pages": 1,
    "web_min_keywords_per_page": 1,
    "web_top_pages_per_issue": 4,
}

DEFAULT_DEF14A = {
    "top_pages_per_category": 3,
    "min_category_score": 1,
    "max_pages_to_select": 30,
    "adjacent_page_threshold": 0.4,
    "use_api_classification": False,
    "api_model": "claude-haiku-4-5-20251001",
}

TESTS = [
    ("3", "top_per_issue=2", {
        "sustainability": {"top_pages_per_issue": 2},
        "def14a": {},
    }),
    ("4", "+table_ind=3", {
        "sustainability": {"top_pages_per_issue": 2, "top_table_indicator_pages": 3},
        "def14a": {},
    }),
    ("5", "+gap=0.5", {
        "sustainability": {"top_pages_per_issue": 2, "top_table_indicator_pages": 3, "score_gap_ratio": 0.5},
        "def14a": {},
    }),
    ("6", "+no_adjacent", {
        "sustainability": {"top_pages_per_issue": 2, "top_table_indicator_pages": 3, "score_gap_ratio": 0.5, "adjacent_page_threshold": 0},
        "def14a": {},
    }),
    ("7", "+def14a=2", {
        "sustainability": {"top_pages_per_issue": 2, "top_table_indicator_pages": 3, "score_gap_ratio": 0.5, "adjacent_page_threshold": 0},
        "def14a": {"top_pages_per_category": 2},
    }),
    ("8", "+max=35", {
        "sustainability": {"top_pages_per_issue": 2, "top_table_indicator_pages": 3, "score_gap_ratio": 0.5, "adjacent_page_threshold": 0, "max_pages_to_select": 35},
        "def14a": {"top_pages_per_category": 2},
    }),
]

# ── Run all tests ──
for suffix, label, overrides in TESTS:
    print(f"\n{'#'*80}")
    print(f"# TEST {suffix}: {label}")
    print(f"{'#'*80}")

    # Apply config overrides
    SUSTAINABILITY_CONFIG.clear()
    SUSTAINABILITY_CONFIG.update(DEFAULT_SUSTAINABILITY)
    SUSTAINABILITY_CONFIG.update(overrides.get("sustainability", {}))

    DEF14A_CONFIG.clear()
    DEF14A_CONFIG.update(DEFAULT_DEF14A)
    DEF14A_CONFIG.update(overrides.get("def14a", {}))

    # Print active config
    print(f"\n   Sustainability config:")
    for k, v in SUSTAINABILITY_CONFIG.items():
        default_v = DEFAULT_SUSTAINABILITY.get(k)
        marker = " ⚡" if v != default_v else ""
        print(f"      {k}: {v}{marker}")
    print(f"   DEF14A config:")
    for k, v in DEF14A_CONFIG.items():
        default_v = DEFAULT_DEF14A.get(k)
        marker = " ⚡" if v != default_v else ""
        print(f"      {k}: {v}{marker}")

    # Set output/log dirs for this test
    test_output_dir = DRIVE_BASE / "ESG Data" / f"filtered_documents_{suffix}"
    test_log_dir = DRIVE_BASE / "ESG Data" / f"filtering_logs_{suffix}"
    test_output_dir.mkdir(parents=True, exist_ok=True)
    test_log_dir.mkdir(parents=True, exist_ok=True)

    # Run pipeline
    for company in TEST_COMPANIES:
        company_log = filter_company_documents(
            company=company,
            input_dir=INPUT_DIR,
            output_dir=test_output_dir,
            log_dir=test_log_dir,
        )

    print(f"\n   ✅ Test {suffix} complete → logs in filtering_logs_{suffix}/")

# ── Restore defaults ──
SUSTAINABILITY_CONFIG.clear()
SUSTAINABILITY_CONFIG.update(DEFAULT_SUSTAINABILITY)
DEF14A_CONFIG.clear()
DEF14A_CONFIG.update(DEFAULT_DEF14A)

print(f"\n{'#'*80}")
print(f"# ALL TESTS COMPLETE — configs restored to defaults")
print(f"{'#'*80}")


################################################################################
# TEST 3: top_per_issue=2
################################################################################

   Sustainability config:
      top_pages_per_issue: 2 ⚡
      min_issue_score: 2
      top_table_indicator_pages: 5
      max_pages_to_select: 50
      score_gap_ratio: 0.25
      adjacent_page_threshold: 0.4
      cdp_context_pages: 1
      web_min_keywords_per_page: 1
      web_top_pages_per_issue: 4
   DEF14A config:
      top_pages_per_category: 3
      min_category_score: 1
      max_pages_to_select: 30
      adjacent_page_threshold: 0.4
      use_api_classification: False
      api_model: claude-haiku-4-5-20251001

📁 FILTERING: Advanced_Micro_Devices

   📄 Sustainability PDFs: 9
      Processing: Report_Advanced_Micro_Devices_2020_1.pdf
         📋 Detected doc type: standard
         25 → 14 pages → 6.53 MB
      Processing: Report_Advanced_Micro_Devices_2020_90.pdf
         📋 Detected doc ty

In [33]:
# Compare all runs for AMD
all_log_dirs = [DRIVE_BASE / "ESG Data" / f"filtering_logs_{i}" for i in [0,3,4,5,6,7,8]]
all_labels = ["Baseline", "top_per=2", "+tbl=3", "+gap=0.5", "+no_adj", "+def14a=2", "+max=35"]

compare_pipeline_runs("Advanced_Micro_Devices", all_log_dirs, all_labels)
compare_pipeline_runs("Nvidia", all_log_dirs, all_labels)


PIPELINE RUN COMPARISON: Advanced_Micro_Devices

📋 Configuration Comparison:
--------------------------------------------------------------------------------

   DEF14A:
      adjacent_page_threshold        Baseline: 0.4 | top_per=2: 0.4 | +tbl=3: 0.4 | +gap=0.5: 0.4 | +no_adj: 0.4 | +def14a=2: 0.4 | +max=35: 0.4
      api_model                      Baseline: claude-haiku-4-5-20251001 | top_per=2: claude-haiku-4-5-20251001 | +tbl=3: claude-haiku-4-5-20251001 | +gap=0.5: claude-haiku-4-5-20251001 | +no_adj: claude-haiku-4-5-20251001 | +def14a=2: claude-haiku-4-5-20251001 | +max=35: claude-haiku-4-5-20251001
      max_pages_to_select            Baseline: 30 | top_per=2: 30 | +tbl=3: 30 | +gap=0.5: 30 | +no_adj: 30 | +def14a=2: 30 | +max=35: 30
      min_category_score             Baseline: 1 | top_per=2: 1 | +tbl=3: 1 | +gap=0.5: 1 | +no_adj: 1 | +def14a=2: 1 | +max=35: 1
   ⚡ top_pages_per_category         Baseline: 3 | top_per=2: 3 | +tbl=3: 3 | +gap=0.5: 3 | +no_adj: 3 | +def14a=2: 2

{'labels': ['Baseline',
  'top_per=2',
  '+tbl=3',
  '+gap=0.5',
  '+no_adj',
  '+def14a=2',
  '+max=35'],
 'total_filtered': {'Baseline': 478,
  'top_per=2': 455,
  '+tbl=3': 436,
  '+gap=0.5': 429,
  '+no_adj': 429,
  '+def14a=2': 397,
  '+max=35': 364},
 'total_original': {'Baseline': 1207,
  'top_per=2': 1207,
  '+tbl=3': 1207,
  '+gap=0.5': 1207,
  '+no_adj': 1207,
  '+def14a=2': 1207,
  '+max=35': 1207}}

In [34]:
# =============================================================================
# RECALL COMPARISON ACROSS RUNS
# =============================================================================

def compare_recall_across_runs(validators: Dict[str, 'FilteringValidator'],
                                log_dirs: List[Path],
                                run_labels: List[str]):
    """
    Run the filtering validator against multiple pipeline runs and compare recall.

    Args:
        validators: Dict of {company: FilteringValidator} with expected pages already added
        log_dirs: List of log directory paths
        run_labels: Labels for each run
    """
    print(f"\n{'='*80}")
    print(f"RECALL COMPARISON ACROSS RUNS")
    print(f"{'='*80}")

    # Header
    header = f"{'Document':<55}" + "".join([f"{label:>10}" for label in run_labels])
    print(f"\n{header}")
    print(f"{'-'*len(header)}")

    for company, validator in validators.items():
        print(f"\n📁 {company}")

        company_recall_per_run = []

        for label, log_dir in zip(run_labels, log_dirs):
            # Reset pipeline pages and reload from this run's log
            validator.pipeline_pages = {}
            validator.load_pipeline_results(log_dir)

        # Now compare per document across all runs
        for filename, expected_data in validator.expected_pages.items():
            expected = set(expected_data['pages'])
            row = f"   {filename:<52}"

            for label, log_dir in zip(run_labels, log_dirs):
                # Reload for this run
                validator.pipeline_pages = {}
                loaded = validator.load_pipeline_results(log_dir)

                if not loaded:
                    row += f"{'N/A':>10}"
                    continue

                pipeline = set(validator.pipeline_pages.get(filename, []))
                found = len(expected & pipeline)
                recall = found / len(expected) * 100 if expected else 100
                missed = len(expected) - found

                if recall == 100:
                    row += f"{'✅ 100%':>10}"
                else:
                    row += f"{'⚠️ ' + f'{recall:.0f}%':>10}"

            # Suppress per-doc print from load_pipeline_results
            print(row)

        # Overall recall per run
        row = f"   {'OVERALL':<52}"
        for label, log_dir in zip(run_labels, log_dirs):
            validator.pipeline_pages = {}
            validator.load_pipeline_results(log_dir)

            total_expected = 0
            total_found = 0
            total_pipeline = 0
            for filename, expected_data in validator.expected_pages.items():
                expected = set(expected_data['pages'])
                pipeline = set(validator.pipeline_pages.get(filename, []))
                total_expected += len(expected)
                total_found += len(expected & pipeline)
                total_pipeline += len(pipeline)

            recall = total_found / total_expected * 100 if total_expected > 0 else 0
            row += f"{f'{recall:.1f}%':>10}"

        print(f"\n{row}")

        # Also show total page count per run
        row = f"   {'TOTAL PAGES':<52}"
        for label, log_dir in zip(run_labels, log_dirs):
            validator.pipeline_pages = {}
            validator.load_pipeline_results(log_dir)
            total = sum(len(pages) for pages in validator.pipeline_pages.values())
            row += f"{total:>10}"
        print(row)

                # Total original (unfiltered) pages
        row_orig = f"   {'ORIGINAL PAGES':<52}"
        row_reduction = f"   {'REDUCTION %':<52}"
        for label, log_dir in zip(run_labels, log_dirs):
            log_file = Path(log_dir) / f"{company}_filtering_log.json"
            if log_file.exists():
                with open(log_file, 'r') as f:
                    log = json.load(f)
                original = log.get('summary', {}).get('total_pages_original', 0)
                filtered = log.get('summary', {}).get('total_pages_filtered', 0)
                reduction = (1 - filtered / original) * 100 if original > 0 else 0
                row_orig += f"{original:>10}"
                row_reduction += f"{reduction:>9.1f}%"
            else:
                row_orig += f"{'N/A':>10}"
                row_reduction += f"{'N/A':>10}"
        print(row_orig)
        print(row_reduction)


    print()


print("✅ Recall comparison tool defined")

✅ Recall comparison tool defined


| Test | Folder Suffix | Label | Changes from Baseline |
|------|--------------|-------|----------------------|
| 1 | `_1` | Baseline | V10 initial parameters |
| 2 | `_2` | TOC Removed | Removed 'table of contents' from exclusion keywords |
| 3 | `_3` | top_per_issue=2 | `top_pages_per_issue`: 3→2 |
| 4 | `_4` | +table_ind=3 | Test 3 + `top_table_indicator_pages`: 5→3 |
| 5 | `_5` | +gap=0.5 | Test 4 + `score_gap_ratio`: 0.25→0.5 |
| 6 | `_6` | +no_adjacent | Test 5 + `adjacent_page_threshold`: 0.4→0 (disabled) |
| 7 | `_7` | +def14a=2 | Test 6 + DEF14A `top_pages_per_category`: 3→2 |
| 8 | `_8` | +max=35 | Test 7 + `max_pages_to_select`: 50→35 |


In [41]:
# Assumes validator_amd and validator_nvidia already have add_expected() calls done

all_log_dirs = [DRIVE_BASE / "ESG Data" / f"filtering_logs_{i}" for i in [0,3,4,5,6,7,8]]
all_labels = ["Baseline", "top_per=2", "+tbl=3", "+gap=0.5", "+no_adj", "+def14a=2", "+max=35"]

# Suppress the load prints
import io, contextlib

validators = {
    "Advanced_Micro_Devices": validator_Advanced_Micro_Devices,
    "Nvidia": validator_Nvidia,
}

compare_recall_across_runs(validators, all_log_dirs, all_labels)


RECALL COMPARISON ACROSS RUNS

Document                                                 Baseline top_per=2    +tbl=3  +gap=0.5   +no_adj +def14a=2   +max=35
-----------------------------------------------------------------------------------------------------------------------------

📁 Advanced_Micro_Devices
   Report_Advanced_Micro_Devices_2020_1.pdf                ✅ 100%    ✅ 100%    ✅ 100%    ✅ 100%    ✅ 100%    ✅ 100%    ✅ 100%
   Report_Advanced_Micro_Devices_2020_90.pdf               ⚠️ 92%    ⚠️ 92%    ⚠️ 92%    ⚠️ 85%    ⚠️ 85%    ⚠️ 85%    ⚠️ 85%
   Report_Advanced_Micro_Devices_2021_5.pdf                ✅ 100%    ✅ 100%    ✅ 100%    ✅ 100%    ✅ 100%    ✅ 100%    ✅ 100%
   Report_Advanced_Micro_Devices_2021_92.pdf               ✅ 100%    ✅ 100%    ✅ 100%    ⚠️ 91%    ⚠️ 91%    ⚠️ 91%    ⚠️ 91%
   Report_Advanced_Micro_Devices_2022_12.pdf               ✅ 100%    ✅ 100%    ✅ 100%    ✅ 100%    ✅ 100%    ✅ 100%    ✅ 100%
   Report_Advanced_Micro_Devices_2022_95.pdf               ✅

## Testing and Debugging

### Filter Validation Tool

In [ ]:
import pandas as pd

In [ ]:
# =============================================================================
# PAGE FILTERING VALIDATION TOOL
# =============================================================================

class FilteringValidator:
    """
    Tool to compare manual page selection with automated filtering.

    Usage:
    1. Create validator: validator = FilteringValidator("Advanced_Micro_Devices")
    2. Add expected pages: validator.add_expected("Report_AMD_2024.pdf", [5, 12, 45, 67])
    3. Run comparison: validator.compare_with_pipeline(INPUT_DIR, OUTPUT_DIR)
    """

    def __init__(self, company: str):
        self.company = company
        self.expected_pages = {}  # filename -> list of page numbers
        self.pipeline_pages = {}  # filename -> list of page numbers

    def add_expected(self, filename: str, pages: List[int], notes: str = ""):
        """
        Add expected pages for a document.

        Args:
            filename: PDF filename (e.g., "Report_Advanced_Micro_Devices_2024_91.pdf")
            pages: List of 1-indexed page numbers that should be selected
            notes: Optional notes about why these pages are important
        """
        self.expected_pages[filename] = {
            'pages': sorted(pages),
            'notes': notes
        }
        print(f"   Added: {filename} → {len(pages)} expected pages")

    def load_pipeline_results(self, log_dir: Path):
        """Load filtering results from pipeline 1 logs."""
        log_dir = Path(log_dir)
        log_file = log_dir / f"{self.company}_filtering_log.json"

        if not log_file.exists():
            print(f"   ⚠️ Log file not found: {log_file}")
            return False

        with open(log_file, 'r') as f:
            log_data = json.load(f)

        # Extract selected pages from each document type
        for doc_type in ['sustainability_pdfs', 'def14a_pdfs', 'tenk_pdfs']:
            if doc_type in log_data:
                for doc in log_data[doc_type]:
                    filename = doc.get('filename', '')
                    pages = doc.get('selected_pages', [])
                    self.pipeline_pages[filename] = pages

        #print(f"   Loaded pipeline results for {len(self.pipeline_pages)} documents")
        return True

    def compare(self) -> Dict:
        """
        Compare expected vs pipeline-selected pages.

        Returns detailed comparison report.
        """
        report = {
            'company': self.company,
            'documents': [],
            'summary': {
                'total_expected': 0,
                'total_found': 0,
                'total_missed': 0,
                'total_extra': 0,
                'recall': 0,
                'precision': 0
            }
        }

        all_expected = 0
        all_found = 0
        all_pipeline = 0

        print(f"\n{'='*70}")
        print(f"FILTERING COMPARISON: {self.company}")
        print(f"{'='*70}")

        for filename, expected_data in self.expected_pages.items():
            expected = set(expected_data['pages'])
            pipeline = set(self.pipeline_pages.get(filename, []))

            found = expected & pipeline  # Intersection
            missed = expected - pipeline  # In expected but not in pipeline
            extra = pipeline - expected   # In pipeline but not expected

            all_expected += len(expected)
            all_found += len(found)
            all_pipeline += len(pipeline)

            doc_report = {
                'filename': filename,
                'expected_count': len(expected),
                'pipeline_count': len(pipeline),
                'found_count': len(found),
                'missed_count': len(missed),
                'extra_count': len(extra),
                'missed_pages': sorted(missed),
                'extra_pages': sorted(extra)[:10],  # Limit for readability
                'recall': len(found) / len(expected) * 100 if expected else 100,
                'notes': expected_data.get('notes', '')
            }
            report['documents'].append(doc_report)

            # Print document summary
            status = "✅" if len(missed) == 0 else "⚠️"
            print(f"\n{status} {filename}")
            print(f"   Expected: {len(expected)} | Pipeline: {len(pipeline)} | Found: {len(found)}")

            if missed:
                print(f"   ❌ MISSED pages: {sorted(missed)}")
            if len(extra) > 10:
                print(f"   ➕ Extra pages: {sorted(extra)[:10]}... (+{len(extra)-10} more)")
            elif extra:
                print(f"   ➕ Extra pages: {sorted(extra)}")

            print(f"   Recall: {doc_report['recall']:.1f}%")

        # Overall summary
        recall = all_found / all_expected * 100 if all_expected > 0 else 0
        precision = all_found / all_pipeline * 100 if all_pipeline > 0 else 0

        report['summary'] = {
            'total_expected': all_expected,
            'total_found': all_found,
            'total_missed': all_expected - all_found,
            'total_pipeline': all_pipeline,
            'total_extra': all_pipeline - all_found,
            'recall': recall,
            'precision': precision,
            'f1_score': 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        }

        print(f"\n{'='*70}")
        print(f"SUMMARY")
        print(f"{'='*70}")
        print(f"Total expected pages: {all_expected}")
        print(f"Total pipeline pages: {all_pipeline}")
        print(f"Correctly found: {all_found}")
        print(f"Missed (false negative): {all_expected - all_found}")
        print(f"Extra (false positive): {all_pipeline - all_found}")
        print(f"\nRecall: {recall:.1f}% (of expected pages, how many did we find?)")
        print(f"Precision: {precision:.1f}% (of selected pages, how many were correct?)")
        print(f"F1 Score: {report['summary']['f1_score']:.1f}%")

        return report

    def export_report(self, output_path: str):
        """Export comparison report to JSON."""
        report = self.compare()
        with open(output_path, 'w') as f:
            json.dump(report, f, indent=2)
        print(f"\n✅ Report saved to: {output_path}")
        return report


def create_expected_pages_template(company: str, input_dir: Path) -> str:
    """
    Generate a template for manually entering expected pages.

    Returns Python code that you can fill in.
    """
    input_dir = Path(input_dir)
    company_dir = input_dir / company

    if not company_dir.exists():
        return f"# Company directory not found: {company_dir}"

    template_lines = [
        f"# =============================================================================",
        f"# EXPECTED PAGES FOR: {company}",
        f"# =============================================================================",
        f"# Instructions:",
        f"# 1. Open each PDF and identify pages with ESG data tables",
        f"# 2. Fill in the page numbers below (1-indexed)",
        f"# 3. Add notes about what data is on those pages",
        f"# =============================================================================",
        f"",
        f"validator = FilteringValidator('{company}')",
        f"",
    ]

    # List sustainability PDFs
    pdf_files = sorted([f for f in company_dir.glob("*.pdf")
                       if not any(x in f.name for x in ['10-K', '10K', 'DEF_14A', 'DEF14A'])])

    template_lines.append("# --- SUSTAINABILITY REPORTS ---")
    for pdf in pdf_files:
        template_lines.append(f"validator.add_expected(")
        template_lines.append(f"    '{pdf.name}',")
        template_lines.append(f"    pages=[],  # TODO: Add page numbers")
        template_lines.append(f"    notes=''   # e.g., 'GHG table on p.45, Safety data on p.67'")
        template_lines.append(f")")
        template_lines.append("")

    # List DEF 14A PDFs
    def14a_files = sorted(list(company_dir.glob("*DEF_14A*.pdf")) +
                          list(company_dir.glob("*DEF14A*.pdf")))

    template_lines.append("# --- DEF 14A FILINGS ---")
    for pdf in def14a_files:
        template_lines.append(f"validator.add_expected(")
        template_lines.append(f"    '{pdf.name}',")
        template_lines.append(f"    pages=[],  # TODO: Add page numbers")
        template_lines.append(f"    notes=''   # e.g., 'Board diversity matrix on p.12'")
        template_lines.append(f")")
        template_lines.append("")

    # List 10-K PDFs
    tenk_files = sorted(list(company_dir.glob("*10-K*.pdf")) +
                        list(company_dir.glob("*10K*.pdf")))

    template_lines.append("# --- 10-K FILINGS ---")
    for pdf in tenk_files:
        template_lines.append(f"validator.add_expected(")
        template_lines.append(f"    '{pdf.name}',")
        template_lines.append(f"    pages=[],  # TODO: Add page numbers")
        template_lines.append(f"    notes=''   # e.g., 'Tax rate reconciliation on p.85, Employee count on p.12'")
        template_lines.append(f")")
        template_lines.append("")


    # Add comparison code
    template_lines.extend([
        "# --- RUN COMPARISON ---",
        f"validator.load_pipeline_results(LOG_DIR)",
        f"report = validator.compare()",
        f"# validator.export_report(str(LOG_DIR / '{company}_validation.json'))",
    ])

    return '\n'.join(template_lines)


print("✅ Validation tools defined")

✅ Validation tools defined


In [37]:
# =============================================================================
# USAGE EXAMPLE
# =============================================================================

# =============================================================================
# STEP 1: Generate template for manual page entry
# =============================================================================

template = create_expected_pages_template("Advanced_Micro_Devices", INPUT_DIR)
print(template)


# =============================================================================
# EXPECTED PAGES FOR: Advanced_Micro_Devices
# =============================================================================
# Instructions:
# 1. Open each PDF and identify pages with ESG data tables
# 2. Fill in the page numbers below (1-indexed)
# 3. Add notes about what data is on those pages
# =============================================================================

validator = FilteringValidator('Advanced_Micro_Devices')

# --- SUSTAINABILITY REPORTS ---
validator.add_expected(
    'Report_Advanced_Micro_Devices_2020_1.pdf',
    pages=[],  # TODO: Add page numbers
    notes=''   # e.g., 'GHG table on p.45, Safety data on p.67'
)

validator.add_expected(
    'Report_Advanced_Micro_Devices_2020_90.pdf',
    pages=[],  # TODO: Add page numbers
    notes=''   # e.g., 'GHG table on p.45, Safety data on p.67'
)

validator.add_expected(
    'Report_Advanced_Micro_Devices_2021_5.pdf',
    pages=[],  # TOD

#### AMD Test

In [38]:
# =============================================================================
# STEP 2: Fill in expected pages and run
# =============================================================================
# EXPECTED PAGES FOR: Advanced_Micro_Devices
# Instructions:
# 1. Open each PDF and identify pages with ESG data tables
# 2. Fill in the page numbers below (1-indexed)
# 3. Add notes about what data is on those pages
# =============================================================================


validator_Advanced_Micro_Devices = FilteringValidator('Advanced_Micro_Devices')

# ─── SUSTAINABILITY REPORTS ───────────────────────────────────────────
validator_Advanced_Micro_Devices.add_expected(
    'Report_Advanced_Micro_Devices_2020_1.pdf',
    pages=[18],
    notes=''
)
validator_Advanced_Micro_Devices.add_expected(
    'Report_Advanced_Micro_Devices_2020_90.pdf',
    pages=[44,45,55,58,59,60,61,62,63,64,65,66,67],
    notes=''
)
validator_Advanced_Micro_Devices.add_expected(
    'Report_Advanced_Micro_Devices_2021_5.pdf',
    pages=[20],
    notes=''
)
validator_Advanced_Micro_Devices.add_expected(
    'Report_Advanced_Micro_Devices_2021_92.pdf',
    pages=[80,82,83,84,85,86,87,88,89,90,91],
    notes=''
)
validator_Advanced_Micro_Devices.add_expected(
    'Report_Advanced_Micro_Devices_2022_12.pdf',
    pages=[15,26,27,28,29,33],
    notes=''
)

# Was added later
validator_Advanced_Micro_Devices.add_expected(
    'Report_Advanced_Micro_Devices_2022_95.pdf',
    pages=[63,64],
    notes=''
)

validator_Advanced_Micro_Devices.add_expected(
    'Report_Advanced_Micro_Devices_2023_94.pdf',
    pages=[140,141,149,150],
    notes=''
)
validator_Advanced_Micro_Devices.add_expected(
    'Report_Advanced_Micro_Devices_2024_91.pdf',
    pages=[93,94],
    notes=''
)
validator_Advanced_Micro_Devices.add_expected(
    'Report_Advanced_Micro_Devices_2024_93.pdf',
    pages=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,17,18,19,23,24,26,27,28],
    notes=''
)


# ─── DEF 14A FILINGS ──────────────────────────────────────────────────
validator_Advanced_Micro_Devices.add_expected(
    'SEC_DEF_14A_Advanced_Micro_Devices_2020.pdf',
    pages=[20,22,23,40,75],  # TODO: Add page numbers
    notes='~20+ = Board composition; 22 = Code of Ethics ; 23 = board attendance; 40 = say on pay; 75 = pay ratio'
)
validator_Advanced_Micro_Devices.add_expected(
    'SEC_DEF_14A_Advanced_Micro_Devices_2021.pdf',
    pages=[23,26,28,29,51,86],  # TODO: Add page numbers
    notes='~20+ = Board composition; 28 = Code of Ethics; 29 = board attendandce ; 51 = say on pay; 86 = pay ratio'
)
validator_Advanced_Micro_Devices.add_expected(
    'SEC_DEF_14A_Advanced_Micro_Devices_2022.pdf',
    pages=[25,28,29,53,90],  # TODO: Add page numbers
    notes='~20+ = Board composition; 28 = Code of Ethics; 29 = board attendandce; 53 = say on pay; 90 = pay ratio'
)
validator_Advanced_Micro_Devices.add_expected(
    'SEC_DEF_14A_Advanced_Micro_Devices_2023.pdf',
    pages=[22,25,28,29,52,89],  # TODO: Add page numbers
    notes='~20+ = Board composition; 28 = Code of Ethics; 29 = board attendandce; 52 = say on pay; 75 = pay ratio'
)
validator_Advanced_Micro_Devices.add_expected(
    'SEC_DEF_14A_Advanced_Micro_Devices_2024.pdf',
    pages=[21,25,28,29,52,89],  # TODO: Add page numbers
    notes='~20+ = Board composition; 28 = Code of Ethics; 29 = board attendandce; 52 = say on pay; 89 = pay ratio'
)


# ─── 10-K FILINGS ─────────────────────────────────────────────────────
validator_Advanced_Micro_Devices.add_expected(
    'SEC_10-K_Advanced_Micro_Devices_2020.pdf',
    pages=[15,52,78],  # TODO: Add page numbers
    notes='15 = employee count, 52 = tax rate, 78 = country-by-country'
)
validator_Advanced_Micro_Devices.add_expected(
    'SEC_10-K_Advanced_Micro_Devices_2021.pdf',
    pages=[14,47,73,77],  # TODO: Add page numbers
    notes='14 = employee count, 47/73 = tax rate, 77 = country-by-country'
)
validator_Advanced_Micro_Devices.add_expected(
    'SEC_10-K_Advanced_Micro_Devices_2022.pdf',
    pages=[15,51,69,86],  # TODO: Add page numbers
    notes='15 = employee count, 51/86 = tax rate, 69 = country-by-country'
)
validator_Advanced_Micro_Devices.add_expected(
    'SEC_10-K_Advanced_Micro_Devices_2023.pdf',
    pages=[16,54,72,88],  # TODO: Add page numbers
    notes='16 = employee count, 54/88 = tax rate, 72 = country-by-country'
)
validator_Advanced_Micro_Devices.add_expected(
    'SEC_10-K_Advanced_Micro_Devices_2024.pdf',
    pages=[14,53,72,85],  # TODO: Add page numbers
    notes='14 = employee count, 53/85 = tax rate, 72 = country-by-country'
)


   Added: Report_Advanced_Micro_Devices_2020_1.pdf → 1 expected pages
   Added: Report_Advanced_Micro_Devices_2020_90.pdf → 13 expected pages
   Added: Report_Advanced_Micro_Devices_2021_5.pdf → 1 expected pages
   Added: Report_Advanced_Micro_Devices_2021_92.pdf → 11 expected pages
   Added: Report_Advanced_Micro_Devices_2022_12.pdf → 6 expected pages
   Added: Report_Advanced_Micro_Devices_2022_95.pdf → 2 expected pages
   Added: Report_Advanced_Micro_Devices_2023_94.pdf → 4 expected pages
   Added: Report_Advanced_Micro_Devices_2024_91.pdf → 2 expected pages
   Added: Report_Advanced_Micro_Devices_2024_93.pdf → 22 expected pages
   Added: SEC_DEF_14A_Advanced_Micro_Devices_2020.pdf → 5 expected pages
   Added: SEC_DEF_14A_Advanced_Micro_Devices_2021.pdf → 6 expected pages
   Added: SEC_DEF_14A_Advanced_Micro_Devices_2022.pdf → 5 expected pages
   Added: SEC_DEF_14A_Advanced_Micro_Devices_2023.pdf → 6 expected pages
   Added: SEC_DEF_14A_Advanced_Micro_Devices_2024.pdf → 6 expected p

In [ ]:
# ─── RUN COMPARISON ───────────────────────────────────────────────────
validator_Advanced_Micro_Devices.load_pipeline_results(str(LOG_DIR)+"_0")
report_Advanced_Micro_Devices = validator_Advanced_Micro_Devices.compare()


FILTERING COMPARISON: Advanced_Micro_Devices

✅ Report_Advanced_Micro_Devices_2020_1.pdf
   Expected: 1 | Pipeline: 18 | Found: 1
   ➕ Extra pages: [4, 5, 6, 7, 11, 12, 13, 14, 16, 17]... (+7 more)
   Recall: 100.0%

⚠️ Report_Advanced_Micro_Devices_2020_90.pdf
   Expected: 13 | Pipeline: 35 | Found: 12
   ❌ MISSED pages: [45]
   ➕ Extra pages: [1, 2, 3, 25, 26, 27, 28, 29, 30, 31]... (+13 more)
   Recall: 92.3%

✅ Report_Advanced_Micro_Devices_2021_5.pdf
   Expected: 1 | Pipeline: 19 | Found: 1
   ➕ Extra pages: [3, 4, 5, 6, 8, 9, 11, 12, 13, 14]... (+8 more)
   Recall: 100.0%

✅ Report_Advanced_Micro_Devices_2021_92.pdf
   Expected: 11 | Pipeline: 43 | Found: 11
   ➕ Extra pages: [2, 3, 9, 10, 11, 12, 13, 37, 38, 39]... (+22 more)
   Recall: 100.0%

✅ Report_Advanced_Micro_Devices_2022_12.pdf
   Expected: 6 | Pipeline: 46 | Found: 6
   ➕ Extra pages: [1, 2, 3, 4, 5, 6, 7, 10, 11, 12]... (+30 more)
   Recall: 100.0%

✅ Report_Advanced_Micro_Devices_2022_95.pdf
   Expected: 2 | Pipeli

#### Nvidia Test

In [39]:
validator_Nvidia = FilteringValidator('Nvidia')

# ─── SUSTAINABILITY REPORTS ───────────────────────────────────────────
validator_Nvidia.add_expected(
    'Report_Nvidia_2020_1.pdf',
    pages=[65,66,67,69,70,71,72],
    notes=''
)
validator_Nvidia.add_expected(
    'Report_Nvidia_2021_3.pdf',
    pages=[68,69,70,71,73,74,75],
    notes=''
)
validator_Nvidia.add_expected(
    'Report_Nvidia_2021_11.pdf',
    pages=[2,4,5,27,30,41,42,43,48,54,55],
    notes=''
)
validator_Nvidia.add_expected(
    'Report_Nvidia_2022_7.pdf',
    pages=[29,38,56,57,58,67,72,73],
    notes=''
)
validator_Nvidia.add_expected(
    'Report_Nvidia_2022_8.pdf',
    pages=[22,24,52,54,55,56],
    notes=''
)

validator_Nvidia.add_expected(
    'Report_Nvidia_2022_10.pdf',
    pages=[1],
    notes=''
)

validator_Nvidia.add_expected(
    'Report_Nvidia_2023_2.pdf',
    pages=[23,26,37,38,39,40,48],
    notes=''
)
validator_Nvidia.add_expected(
    'Report_Nvidia_2023_13.pdf',
    pages=[1],
    notes=''
)
validator_Nvidia.add_expected(
    'Report_Nvidia_2024_15.pdf',
    pages=[16,29,30,31,32,33,36,38],
    notes=''
)
# 2025 Document
# validator_Nvidia.add_expected(
#     'Report_Nvidia_2025_18.pdf',
#     pages=[16,29,30,31,32,33,36,38],
#     notes=''
# )


# ─── DEF 14A FILINGS ──────────────────────────────────────────────────
validator_Nvidia.add_expected(
    'SEC_DEF_14A_Nvidia_2020.pdf',
    pages=[7,9,11,18,33,50,68],  # TODO: Add page numbers
    notes='7/9 = board and independence/attendance, 18 = independence, 33 = ethics, 33 = attendance, 11/50 = say on pay (2019), 68 = pay ratio'
)
validator_Nvidia.add_expected(
    'SEC_DEF_14A_Nvidia_2021.pdf',
    pages=[8,9,10,19,22,40,63,78],  # TODO: Add page numbers
    notes='8/9 = board and independence/attendance, 19 = independence, 22 = diversity/women on board, 10/63 = say on pay (2020), 40 = ethics, 78 = pay ratio'
)
validator_Nvidia.add_expected(
    'SEC_DEF_14A_Nvidia_2022.pdf',
    pages=[8,9,10,21,39,62,78],  # TODO: Add page numbers
    notes='8/9 = board and independence/attendance, 21 = diversity/women on board, 10/62 = say on pay, 39 = ethics, 78 = pay ratio'
)
validator_Nvidia.add_expected(
    'SEC_DEF_14A_Nvidia_2023.pdf',
    pages=[8,9,19,32,51,64],  # TODO: Add page numbers
    notes='8/9 = board and independence/attendance, 19 = diversity/women on board, 9/51 = say on pay, 32 = ethics, 64 = pay ratio'
)
validator_Nvidia.add_expected(
    'SEC_DEF_14A_Nvidia_2024.pdf',
    pages=[8,9,21,41,56,72],  # TODO: Add page numbers
    notes='8/9 = board and independence/attendance, 21 = diversity/women on board, 41 = ethics, 56 = say on pay, 72 = pay ratio'
)


# ─── 10-K FILINGS ─────────────────────────────────────────────────────
validator_Nvidia.add_expected(
    'SEC_10-K_Nvidia_2020.pdf',
    pages=[11,37,75,78],  # TODO: Add page numbers
    notes='11 = employee count, 37/75 = tax rate, 78 = country-by-country'
)
validator_Nvidia.add_expected(
    'SEC_10-K_Nvidia_2021.pdf',
    pages=[12,38,77,81],  # TODO: Add page numbers
    notes='12 = employee count, 38/77 = tax rate, 81 = country-by-country'
)
validator_Nvidia.add_expected(
    'SEC_10-K_Nvidia_2022.pdf',
    pages=[13,44,82,86],  # TODO: Add page numbers
    notes='13 = employee count, 44/82= tax rate, 86 = country-by-country'
)
validator_Nvidia.add_expected(
    'SEC_10-K_Nvidia_2023.pdf',
    pages=[12,43,81,85],  # TODO: Add page numbers
    notes='12 = employee count, 43/81 = tax rate (43 components only), 85 = country-by-country'
)
validator_Nvidia.add_expected(
    'SEC_10-K_Nvidia_2024.pdf',
    pages=[11,46,78,82],  # TODO: Add page numbers
    notes='11 = employee count, 46/78 = tax rate, 82 = country-by-country'
)



   Added: Report_Nvidia_2020_1.pdf → 7 expected pages
   Added: Report_Nvidia_2021_3.pdf → 7 expected pages
   Added: Report_Nvidia_2021_11.pdf → 11 expected pages
   Added: Report_Nvidia_2022_7.pdf → 8 expected pages
   Added: Report_Nvidia_2022_8.pdf → 6 expected pages
   Added: Report_Nvidia_2022_10.pdf → 1 expected pages
   Added: Report_Nvidia_2023_2.pdf → 7 expected pages
   Added: Report_Nvidia_2023_13.pdf → 1 expected pages
   Added: Report_Nvidia_2024_15.pdf → 8 expected pages
   Added: SEC_DEF_14A_Nvidia_2020.pdf → 7 expected pages
   Added: SEC_DEF_14A_Nvidia_2021.pdf → 8 expected pages
   Added: SEC_DEF_14A_Nvidia_2022.pdf → 7 expected pages
   Added: SEC_DEF_14A_Nvidia_2023.pdf → 6 expected pages
   Added: SEC_DEF_14A_Nvidia_2024.pdf → 6 expected pages
   Added: SEC_10-K_Nvidia_2020.pdf → 4 expected pages
   Added: SEC_10-K_Nvidia_2021.pdf → 4 expected pages
   Added: SEC_10-K_Nvidia_2022.pdf → 4 expected pages
   Added: SEC_10-K_Nvidia_2023.pdf → 4 expected pages
   Added

In [40]:
# ─── RUN COMPARISON ───────────────────────────────────────────────────
validator_Nvidia.load_pipeline_results(str(LOG_DIR)+"_0")
report_Nvidia = validator_Nvidia.compare()


FILTERING COMPARISON: Nvidia

✅ Report_Nvidia_2020_1.pdf
   Expected: 7 | Pipeline: 47 | Found: 7
   ➕ Extra pages: [11, 12, 13, 17, 18, 19, 24, 25, 26, 31]... (+30 more)
   Recall: 100.0%

✅ Report_Nvidia_2021_3.pdf
   Expected: 7 | Pipeline: 47 | Found: 7
   ➕ Extra pages: [10, 11, 12, 16, 17, 18, 22, 23, 24, 25]... (+30 more)
   Recall: 100.0%

✅ Report_Nvidia_2021_11.pdf
   Expected: 11 | Pipeline: 49 | Found: 11
   ➕ Extra pages: [21, 25, 28, 29, 31, 32, 34, 35, 36, 40]... (+28 more)
   Recall: 100.0%

✅ Report_Nvidia_2022_7.pdf
   Expected: 8 | Pipeline: 50 | Found: 8
   ➕ Extra pages: [4, 6, 8, 13, 23, 25, 26, 30, 32, 34]... (+32 more)
   Recall: 100.0%

✅ Report_Nvidia_2022_8.pdf
   Expected: 6 | Pipeline: 48 | Found: 6
   ➕ Extra pages: [6, 7, 8, 9, 10, 11, 14, 15, 16, 17]... (+32 more)
   Recall: 100.0%

✅ Report_Nvidia_2022_10.pdf
   Expected: 1 | Pipeline: 3 | Found: 1
   ➕ Extra pages: [2, 3]
   Recall: 100.0%

⚠️ Report_Nvidia_2023_2.pdf
   Expected: 7 | Pipeline: 43 | F

In [ ]:
import fitz
import re

pdf_path = str(INPUT_DIR / "Advanced_Micro_Devices" / "SEC_DEF_14A_Advanced_Micro_Devices_2020.pdf")
doc = fitz.open(pdf_path)

# Check page 75 (expected: pay_ratio) and page 20 (expected: board)
for page_num in [20, 40, 75]:
    page = doc[page_num - 1]
    text = page.get_text()
    text_lower = text.lower()

    print(f"\n{'='*60}")
    print(f"PAGE {page_num} — First 500 chars:")
    print(f"{'='*60}")
    print(repr(text[:500]))  # repr shows whitespace/newlines

    # Test regex patterns
    print(f"\n  Regex matches:")
    for cat, data in SEC_DEF14A_PATTERNS.items():
        for p in data['patterns']:
            pattern = re.compile(p, re.IGNORECASE)
            match = pattern.search(text)
            if match:
                print(f"    ✅ {cat}: '{p[:50]}' → matched '{match.group()}'")

    # Test old V9 keywords for comparison
    old_keywords = {
        'board': ['board of directors', 'board composition', 'director independence'],
        'pay_ratio': ['pay ratio', 'ceo pay ratio', 'median employee'],
        'say_on_pay': ['say on pay', 'say-on-pay', 'advisory vote'],
    }
    print(f"\n  V9 keyword matches:")
    for cat, kws in old_keywords.items():
        for kw in kws:
            if kw in text_lower:
                print(f"    ✅ {cat}: '{kw}'")

doc.close()


PAGE 20 — First 500 chars:
'Table of Contents\n\xa0\xa02021 NOTICE OF MEETING AND PROXY STATEMENT\xa0\xa0\n\xa0\n\xa0\nCORPORATE GOVERNANCE\nThe Board has adopted the Governance Principles to address significant corporate governance matters. The Governance\nPrinciples provide a framework for our corporate governance matters and include topics such as Board and Board committee composition\nand evaluation. The Nominating and Corporate Governance Committee is responsible for reviewing the Governance Principles and\nrecommending any changes to the Gove'

  Regex matches:
    ✅ board: 'director\s+nominees?' → matched 'director nominees'
    ✅ board: 'board\s+of\s+directors?' → matched 'board of directors'
    ✅ board: 'director\s+(?:independence|qualifications?)' → matched 'director independence'
    ✅ board: 'independent\s+directors?' → matched 'independent directors'

  V9 keyword matches:
    ✅ board: 'board of directors'
    ✅ board: 'director independence'

PAGE 40 — First 500 chars:


In [ ]:
# Quick check: does the main pipeline actually process these files?
company_input = INPUT_DIR / "Advanced_Micro_Devices"
def14a_pdfs = list(company_input.glob("*DEF_14A*.pdf")) + list(company_input.glob("*DEF14A*.pdf"))
print(f"Found {len(def14a_pdfs)} DEF 14A files:")
for f in sorted(def14a_pdfs):
    print(f"  {f.name}")

# Now test the filter directly on the 2020 file
from pathlib import Path
test_path = company_input / "SEC_DEF_14A_Advanced_Micro_Devices_2020.pdf"
print(f"\nDirect filter test on 2020:")
selected, log = filter_def14a_pdf(str(test_path))
print(f"  Selected pages: {selected}")
print(f"  Error: {log.get('error', 'None')}")
print(f"  Total pages: {log.get('total_pages', 'N/A')}")
print(f"  Category selections: {log.get('category_selections', {})}")


Found 5 DEF 14A files:
  SEC_DEF_14A_Advanced_Micro_Devices_2020.pdf
  SEC_DEF_14A_Advanced_Micro_Devices_2021.pdf
  SEC_DEF_14A_Advanced_Micro_Devices_2022.pdf
  SEC_DEF_14A_Advanced_Micro_Devices_2023.pdf
  SEC_DEF_14A_Advanced_Micro_Devices_2024.pdf

Direct filter test on 2020:
  Selected pages: []
  Error: None
  Total pages: 84
  Category selections: {'board': {'pages': [], 'scores': [], 'matched_patterns': [], 'total_candidates': 0}, 'diversity': {'pages': [], 'scores': [], 'matched_patterns': [], 'total_candidates': 0}, 'meetings': {'pages': [], 'scores': [], 'matched_patterns': [], 'total_candidates': 0}, 'pay_ratio': {'pages': [], 'scores': [], 'matched_patterns': [], 'total_candidates': 0}, 'say_on_pay': {'pages': [], 'scores': [], 'matched_patterns': [], 'total_candidates': 0}, 'ethics': {'pages': [], 'scores': [], 'matched_patterns': [], 'total_candidates': 0}, 'ownership': {'pages': [], 'scores': [], 'matched_patterns': [], 'total_candidates': 0}}


In [ ]:
# Every page in a DEF 14A starts with "Table of Contents" as a running header
doc = fitz.open(str(INPUT_DIR / "Advanced_Micro_Devices" / "SEC_DEF_14A_Advanced_Micro_Devices_2020.pdf"))
toc_count = 0
for i in range(len(doc)):
    text = doc[i].get_text().lower()
    if 'table of contents' in text:
        toc_count += 1
print(f"Pages with 'table of contents': {toc_count} / {len(doc)}")
doc.close()

Pages with 'table of contents': 84 / 84


In [ ]:
# Run this on the 2020 file, page 40 (the missed say-on-pay page)
import fitz
pdf_path = str(INPUT_DIR / "Advanced_Micro_Devices" / "SEC_DEF_14A_Advanced_Micro_Devices_2020.pdf")
doc = fitz.open(pdf_path)

for page_num in [40]:
    page = doc[page_num - 1]
    text = page.get_text()
    print(f"PAGE {page_num} — First 800 chars:")
    print(repr(text[:800]))
    print()

    # Test all say_on_pay patterns
    for p in SEC_DEF14A_PATTERNS['say_on_pay']['patterns']:
        pattern = re.compile(p, re.IGNORECASE)
        match = pattern.search(text)
        if match:
            print(f"  ✅ '{p[:60]}' → '{match.group()}'")
        else:
            print(f"  ❌ '{p[:60]}'")

doc.close()

PAGE 40 — First 800 chars:
'Table of Contents\n\xa0\xa02021 NOTICE OF MEETING AND PROXY STATEMENT\xa0\xa0\n\xa0\nCompensation Discussion and Analysis (continued)\n\xa0\xa0\nThe table below compares our market capitalization to the “realized pay” of our Chief Executive Officer and the average\n“realized pay” of our other Named Executive Officers for each of our last three fiscal years. As this table demonstrates, the compensation\ncollectively “realized” by our Named Executive Officers in the aggregate from the end of fiscal 2018 to the end of fiscal 2020 represented\n0.28% of the $101.2\xa0billion in value created for our stockholders during that same period.\n\xa0\n\xa0\n(1)\nEach Named Executive Officer’s “realized pay” is the sum of his or her earned base salary, actual annual incentive bonus under the EIP, any discretionary or retention\nbonus amounts rea'

  ✅ 'say[\-\s]on[\-\s]pay' → 'Say On Pay'
  ❌ 'advisory\s+(?:vote|approval)\s+(?:on|to\s+approve)\s+(?:exec'
  ❌ 'approv(?:e|a

In [ ]:
doc = fitz.open(str(INPUT_DIR / "Advanced_Micro_Devices" / "SEC_DEF_14A_Advanced_Micro_Devices_2020.pdf"))
text = doc[39].get_text()
# Search for vote-related numbers
print("Full page text:")
print(text)
doc.close()


Full page text:
Table of Contents
  2021 NOTICE OF MEETING AND PROXY STATEMENT  
 
Compensation Discussion and Analysis (continued)
  
The table below compares our market capitalization to the “realized pay” of our Chief Executive Officer and the average
“realized pay” of our other Named Executive Officers for each of our last three fiscal years. As this table demonstrates, the compensation
collectively “realized” by our Named Executive Officers in the aggregate from the end of fiscal 2018 to the end of fiscal 2020 represented
0.28% of the $101.2 billion in value created for our stockholders during that same period.
 
 
(1)
Each Named Executive Officer’s “realized pay” is the sum of his or her earned base salary, actual annual incentive bonus under the EIP, any discretionary or retention
bonus amounts realized, other compensation, and Form W-2 income realized due to stock option exercises and the vesting of restricted stock unit and performance-
based awards under our equity plans for 

In [ ]:
# ------------------------------------------------
# Table Of Contents Check
# ------------------------------------------------

# Check of most pages in a document have "Table of Contents"
import pdfplumber
pdf_path = str(INPUT_DIR / "Nvidia" / "Report_Nvidia_2021_3.pdf")  # fill in the filename
with pdfplumber.open(pdf_path) as pdf:
    toc_count = sum(1 for p in pdf.pages if 'table of contents' in (p.extract_text() or '').lower())
    print(f"Pages with 'table of contents': {toc_count} / {len(pdf.pages)}")

Pages with 'table of contents': 89 / 94


### Test DEF 14A Filtering

In [ ]:
def test_def14a_filtering(company: str, filename: str, input_dir: Path):
    """Test the per-category filtering and show results."""

    pdf_path = Path(input_dir) / company / filename

    if not pdf_path.exists():
        print(f"❌ File not found: {pdf_path}")
        return

    selected_pages, log_data = filter_def14a_pdf(str(pdf_path))

    print(f"{'='*70}")
    print(f"PER-CATEGORY FILTERING: {filename}")
    print(f"{'='*70}")
    print(f"Total pages in PDF: {log_data['total_pages']}")
    print(f"Selected pages: {len(selected_pages)}")
    print(f"\n📊 Selection by Category:")
    print(f"{'-'*50}")

    for category, data in log_data['category_selections'].items():
        pages = data['pages']
        scores = data['scores']
        candidates = data['total_candidates']

        if pages:
            pages_str = ', '.join([f"p{p}({s})" for p, s in zip(pages, scores)])
            print(f"   {category:<12} → {pages_str}")
            print(f"   {'':12}   ({candidates} candidates)")
        else:
            print(f"   {category:<12} → No matches")

    print(f"\n📄 Final Selection: {selected_pages}")

    return selected_pages, log_data




# Usage:
# test_def14a_filtering("Advanced_Micro_Devices", "SEC_DEF_14A_Advanced_Micro_Devices_2020.pdf", INPUT_DIR)

## Expected Output
# ======================================================================
# PER-CATEGORY FILTERING: SEC_DEF_14A_Advanced_Micro_Devices_2020.pdf
# ======================================================================
# Total pages in PDF: 84 | Selected pages: 18

# 📊 Selection by Category:
# --------------------------------------------------
#    board        → p20(4), p17(4), p24(4)
#                    (28 candidates)
#    diversity    → p19(2), p20(1), p18(1)
#                    (5 candidates)
#    meetings     → p23(3), p3(2), p18(1)
#                    (8 candidates)
#    pay_ratio    → p75(3), p4(2), p14(1)
#                    (12 candidates)
#    say_on_pay   → p80(3), p81(3), p84(2)
#                    (25 candidates)
#    ethics       → p22(2), p23(1), p41(1)
#                    (6 candidates)
#    ownership    → p18(2), p31(2), p8(1)
#                    (10 candidates)

# 📄 Final Selection: [3, 4, 8, 14, 17, 18, 19, 20, 22, 23, 24, 31, 41, 75, 80, 81, 84]

In [ ]:
# Usage:
#test_def14a_filtering("Advanced_Micro_Devices", "SEC_DEF_14A_Advanced_Micro_Devices_2020.pdf", INPUT_DIR)
testdef14a2020 = test_def14a_filtering("Advanced_Micro_Devices", "SEC_DEF_14A_Advanced_Micro_Devices_2020.pdf", INPUT_DIR)

PER-CATEGORY FILTERING: SEC_DEF_14A_Advanced_Micro_Devices_2020.pdf
Total pages in PDF: 84
Selected pages: 24

📊 Selection by Category:
--------------------------------------------------
   board        → p17(12), p20(12), p23(12)
                  (25 candidates)
   diversity    → No matches
   meetings     → p23(9), p13(3), p18(3)
                  (6 candidates)
   pay_ratio    → p75(30), p4(12)
                  (2 candidates)
   say_on_pay   → p40(25), p4(10), p80(10)
                  (5 candidates)
   ethics       → p22(6), p23(4), p24(2)
                  (6 candidates)
   ownership    → p30(6), p31(4), p32(4)
                  (15 candidates)

📄 Final Selection: [4, 12, 13, 14, 16, 17, 18, 19, 20, 22, 23, 24, 25, 29, 30, 31, 32, 33, 40, 41, 75, 76, 80, 81]


### Test Sustainability Report Filtering

In [ ]:
# =============================================================================
# TEST FUNCTION FOR SUSTAINABILITY FILTERING (V10)
# =============================================================================

def test_sustainability_filtering(company: str, filename: str, input_dir: Path):
    """Test sustainability filtering on a single document and show detailed results."""

    pdf_path = Path(input_dir) / company / filename

    if not pdf_path.exists():
        print(f"❌ File not found: {pdf_path}")
        return None, None

    selected_pages, log_data = filter_sustainability_pdf(str(pdf_path))

    doc_type = log_data.get('doc_type', 'unknown')

    print(f"{'='*70}")
    print(f"SUSTAINABILITY FILTERING: {filename}")
    print(f"{'='*70}")
    print(f"Document type: {doc_type}")
    print(f"Total pages in PDF: {log_data.get('total_pages', 'N/A')}")
    print(f"Selected pages: {len(selected_pages)}")

    # ── Excluded pages ──
    excluded = log_data.get('excluded_pages', [])
    if excluded:
        print(f"Excluded pages: {len(excluded)} {excluded[:10]}{'...' if len(excluded) > 10 else ''}")

    # ── CDP-specific output ──
    if doc_type == DocType.CDP:
        print(f"\n📊 CDP Section Matches:")
        print(f"{'-'*60}")

        section_matches = log_data.get('section_matches', {})
        for pn in sorted(section_matches.keys()):
            sections = section_matches[pn]
            sec_str = ', '.join([f"{sid}→{issue}" for sid, issue in sections])
            print(f"   p{pn}: {sec_str}")

        print(f"\n📊 Issue Coverage:")
        print(f"{'-'*60}")
        issue_coverage = log_data.get('issue_coverage', {})
        for issue, pages in sorted(issue_coverage.items()):
            print(f"   ✅ {issue:<18} → {len(pages)} pages: {pages[:10]}{'...' if len(pages) > 10 else ''}")

    # ── Standard / Web PDF output ──
    else:
        issue_selections = log_data.get('issue_selections', {})

        issues_with_pages = [(issue, data) for issue, data in issue_selections.items() if data.get('pages')]
        issues_without_pages = [(issue, data) for issue, data in issue_selections.items() if not data.get('pages')]

        print(f"\n📊 Selection by Issue:")
        print(f"{'-'*60}")

        for issue, data in sorted(issues_with_pages, key=lambda x: -max(x[1]['scores']) if x[1]['scores'] else 0):
            pages = data['pages']
            scores = data['scores']
            candidates = data['total_candidates']
            pages_str = ', '.join([f"p{p}({s})" for p, s in zip(pages, scores)])
            print(f"   ✅ {issue:<18} → {pages_str:<40} ({candidates} candidates)")

        if issues_without_pages:
            print(f"\n   ⚠️ No matches:")
            for issue, data in issues_without_pages:
                print(f"      {issue}")

        # Table indicator pages
        table_pages = log_data.get('table_indicator_pages', [])
        if table_pages:
            print(f"\n📋 Table Indicator Pages ({len(table_pages)} added):")
            print(f"{'-'*60}")
            for tp in table_pages:
                print(f"   p{tp['page_num']} (score: {tp['score']})")

    # ── Adjacent pages (all types) ──
    adj_pages = log_data.get('adjacent_pages_added', [])
    if adj_pages:
        print(f"\n📎 Adjacent Pages Added ({len(adj_pages)}): {sorted(adj_pages)}")

    print(f"\n📄 Final Selection ({len(selected_pages)} pages): {selected_pages}")

    return selected_pages, log_data

In [ ]:
# Usage:
# test_sustainability_filtering("Advanced_Micro_Devices", "Report_Advanced_Micro_Devices_2024_91.pdf", INPUT_DIR)
testsustainability = test_sustainability_filtering("Nvidia", "Report_Nvidia_2021_3.pdf", INPUT_DIR)

         📋 Detected doc type: standard
SUSTAINABILITY FILTERING: Report_Nvidia_2021_3.pdf
Document type: standard
Total pages in PDF: 94
Selected pages: 47
Excluded pages: 3 [78, 80, 94]

📊 Selection by Issue:
------------------------------------------------------------
   ✅ energy             → p75(36), p73(28), p68(24)                (16 candidates)
   ✅ water              → p75(36), p82(14)                         (4 candidates)
   ✅ waste              → p75(34), p74(26), p68(23)                (9 candidates)
   ✅ carbon_emissions   → p73(33), p74(33), p68(23)                (15 candidates)
   ✅ ewaste             → p75(32)                                  (6 candidates)
   ✅ health_safety      → p73(32), p68(23)                         (22 candidates)
   ✅ workforce          → p73(29), p74(26), p71(24)                (20 candidates)
   ✅ diversity          → p73(28), p71(24), p24(8)                 (28 candidates)
   ✅ financial          → p68(24), p71(23), p69(19)                (

### Test 10-K Filtering

In [ ]:
# =============================================================================
# 10-K TEST FUNCTION
# =============================================================================

def test_10k_filtering(company: str, filename: str, input_dir: Path):
    """Test 10-K filtering on a single document and show detailed results."""

    pdf_path = Path(input_dir) / company / filename

    if not pdf_path.exists():
        print(f"❌ File not found: {pdf_path}")
        return None, None

    selected_pages, log_data = filter_10k_pdf(str(pdf_path))

    print(f"{'='*70}")
    print(f"10-K FILTERING: {filename}")
    print(f"{'='*70}")
    print(f"Total pages: {log_data.get('total_pages', 'N/A')}")
    print(f"Selected pages: {len(selected_pages)}")

    print(f"\n📊 Selection by Group:")
    print(f"{'-'*60}")

    group_selections = log_data.get('group_selections', {})
    groups_with_pages = [(g, d) for g, d in group_selections.items() if d.get('pages')]
    groups_without = [(g, d) for g, d in group_selections.items() if not d.get('pages')]

    for group, data in sorted(groups_with_pages, key=lambda x: -max(x[1]['scores']) if x[1]['scores'] else 0):
        pages = data['pages']
        scores = data['scores']
        candidates = data['total_candidates']
        pages_str = ', '.join([f"p{p}({s})" for p, s in zip(pages, scores)])
        print(f"   ✅ {group:<22} → {pages_str:<40} ({candidates} candidates)")

    if groups_without:
        print(f"\n   ⚠️ No matches:")
        for group, data in groups_without:
            print(f"      {group}")

    adj_pages = log_data.get('adjacent_pages_added', [])
    if adj_pages:
        print(f"\n📎 Adjacent Pages Added ({len(adj_pages)}): {sorted(adj_pages)}")

    print(f"\n📄 Final Selection ({len(selected_pages)} pages): {selected_pages}")

    return selected_pages, log_data


print("✅ 10-K test function defined")
print("   Usage: test_10k_filtering('Advanced_Micro_Devices', 'SEC_10-K_Advanced_Micro_Devices_2024.pdf', INPUT_DIR)")

✅ 10-K test function defined
   Usage: test_10k_filtering('Advanced_Micro_Devices', 'SEC_10-K_Advanced_Micro_Devices_2024.pdf', INPUT_DIR)


In [ ]:
test_10k_filtering('Advanced_Micro_Devices', 'SEC_10-K_Advanced_Micro_Devices_2023.pdf', INPUT_DIR)

10-K FILTERING: SEC_10-K_Advanced_Micro_Devices_2023.pdf
Total pages: 109
Selected pages: 19

📊 Selection by Group:
------------------------------------------------------------
   ✅ tax                    → p88(55), p69(50), p54(45)                (16 candidates)
   ✅ geographic_revenue     → p72(31), p52(23), p70(23)                (20 candidates)
   ✅ employees              → p16(22), p17(14)                         (2 candidates)
   ✅ accounting             → p1(16), p99(13), p96(11)                 (11 candidates)

📎 Adjacent Pages Added (8): [18, 51, 71, 73, 87, 89, 97, 98]

📄 Final Selection (19 pages): [1, 16, 17, 18, 51, 52, 54, 69, 70, 71, 72, 73, 87, 88, 89, 96, 97, 98, 99]


([1, 16, 17, 18, 51, 52, 54, 69, 70, 71, 72, 73, 87, 88, 89, 96, 97, 98, 99],
 {'filename': 'SEC_10-K_Advanced_Micro_Devices_2023.pdf',
  'total_pages': 109,
  'selected_pages': [1,
   16,
   17,
   18,
   51,
   52,
   54,
   69,
   70,
   71,
   72,
   73,
   87,
   88,
   89,
   96,
   97,
   98,
   99],
  'page_matches': [{'page_num': 1,
    'score': 16.0,
    'groups': ['accounting(16)']},
   {'page_num': 5, 'score': 4.0, 'groups': ['geographic_revenue(4)']},
   {'page_num': 6, 'score': 18.0, 'groups': ['geographic_revenue(18)']},
   {'page_num': 7, 'score': 4.0, 'groups': ['geographic_revenue(4)']},
   {'page_num': 8, 'score': 9.0, 'groups': ['geographic_revenue(9)']},
   {'page_num': 16,
    'score': 36.0,
    'groups': ['geographic_revenue(14)', 'employees(22)']},
   {'page_num': 17, 'score': 14.0, 'groups': ['employees(14)']},
   {'page_num': 18,
    'score': 17.0,
    'groups': ['accounting(8)', 'geographic_revenue(9)']},
   {'page_num': 20, 'score': 5.0, 'groups': ['tax(5)']

In [ ]:
test_10k_filtering('Apple_Inc', 'SEC_10-K_Apple_Inc._2024.pdf', INPUT_DIR)

10-K FILTERING: SEC_10-K_Apple_Inc._2024.pdf
Total pages: 64
Selected pages: 14

📊 Selection by Group:
------------------------------------------------------------
   ✅ tax                    → p45(40), p46(35), p29(30)                (11 candidates)
   ✅ geographic_revenue     → p41(27), p26(23), p30(18)                (8 candidates)
   ✅ accounting             → p2(16), p53(13), p55(11)                 (4 candidates)
   ✅ employees              → p7(13)                                   (1 candidates)

📎 Adjacent Pages Added (4): [8, 44, 52, 56]

📄 Final Selection (14 pages): [2, 7, 8, 26, 29, 30, 41, 44, 45, 46, 52, 53, 55, 56]


([2, 7, 8, 26, 29, 30, 41, 44, 45, 46, 52, 53, 55, 56],
 {'filename': 'SEC_10-K_Apple_Inc._2024.pdf',
  'total_pages': 64,
  'selected_pages': [2, 7, 8, 26, 29, 30, 41, 44, 45, 46, 52, 53, 55, 56],
  'page_matches': [{'page_num': 2,
    'score': 16.0,
    'groups': ['accounting(16)']},
   {'page_num': 5, 'score': 8.0, 'groups': ['geographic_revenue(8)']},
   {'page_num': 7, 'score': 13.0, 'groups': ['employees(13)']},
   {'page_num': 8, 'score': 4.0, 'groups': ['geographic_revenue(4)']},
   {'page_num': 20, 'score': 15.0, 'groups': ['tax(15)']},
   {'page_num': 21, 'score': 10.0, 'groups': ['tax(10)']},
   {'page_num': 22, 'score': 9.0, 'groups': ['geographic_revenue(9)']},
   {'page_num': 26, 'score': 23.0, 'groups': ['geographic_revenue(23)']},
   {'page_num': 29, 'score': 30.0, 'groups': ['tax(30)']},
   {'page_num': 30,
    'score': 38.0,
    'groups': ['tax(20)', 'geographic_revenue(18)']},
   {'page_num': 34, 'score': 15.0, 'groups': ['tax(15)']},
   {'page_num': 39, 'score': 10.

### Pipeline Run Comparison Tool

In [ ]:
# =============================================================================
# PIPELINE RUN COMPARISON TOOL
# =============================================================================

def compare_pipeline_runs(company: str, log_dirs: List[Path], run_labels: List[str] = None):
    """
    Compare filtering results for the same company across multiple pipeline runs.

    Args:
        company: Company name (e.g., "Advanced_Micro_Devices")
        log_dirs: List of log directory paths (e.g., [LOG_DIR_1, LOG_DIR_2, LOG_DIR_3])
        run_labels: Optional labels for each run (e.g., ["baseline", "tight_v1", "tight_v2"])
                    If not provided, uses "Run 1", "Run 2", etc.

    Usage:
        compare_pipeline_runs("Advanced_Micro_Devices", [
            DRIVE_BASE / "ESG Data" / "filtering_logs_1",
            DRIVE_BASE / "ESG Data" / "filtering_logs_2",
        ], run_labels=["V10 baseline", "top_per_issue=2"])
    """
    if run_labels is None:
        run_labels = [f"Run {i+1}" for i in range(len(log_dirs))]

    assert len(log_dirs) == len(run_labels), "log_dirs and run_labels must have same length"

    # ── Load logs ──
    run_data = {}
    for label, log_dir in zip(run_labels, log_dirs):
        log_file = Path(log_dir) / f"{company}_filtering_log.json"
        if not log_file.exists():
            print(f"⚠️ Log not found for '{label}': {log_file}")
            continue
        with open(log_file, 'r') as f:
            run_data[label] = json.load(f)

    if len(run_data) < 2:
        print("❌ Need at least 2 valid runs to compare.")
        return None

    labels = list(run_data.keys())

    # ── Extract per-document page selections ──
    def get_doc_pages(log_data):
        """Extract {filename: set(pages)} from a company log."""
        docs = {}
        for doc_type in ['sustainability_pdfs', 'def14a_pdfs']:
            for doc in log_data.get(doc_type, []):
                filename = doc.get('filename', '')
                pages = set(doc.get('selected_pages', []))
                total = doc.get('total_pages', 0)
                docs[filename] = {'pages': pages, 'total_pages': total}
        # 10-K doesn't have page numbers, skip for comparison
        return docs

    all_runs = {label: get_doc_pages(data) for label, data in run_data.items()}

    # Get union of all filenames across runs
    all_filenames = sorted(set().union(*[set(docs.keys()) for docs in all_runs.values()]))

    # ══════════════════════════════════════════════════════════════════
    # HEADER
    # ══════════════════════════════════════════════════════════════════
    print(f"\n{'='*80}")
    print(f"PIPELINE RUN COMPARISON: {company}")
    print(f"{'='*80}")

    # ── Config comparison ──
    print(f"\n📋 Configuration Comparison:")
    print(f"{'-'*80}")

    # Try to extract configs from summary
    configs_found = False
    for label in labels:
        summary = run_data[label].get('summary', {})
        config = summary.get('config', {})
        if config:
            configs_found = True

    if configs_found:
        # Collect all config keys across all runs
        all_config_types = set()
        for label in labels:
            config = run_data[label].get('summary', {}).get('config', {})
            all_config_types.update(config.keys())

        for config_type in sorted(all_config_types):
            print(f"\n   {config_type.upper()}:")
            # Collect all param keys for this config type
            all_params = set()
            for label in labels:
                config = run_data[label].get('summary', {}).get('config', {}).get(config_type, {})
                all_params.update(config.keys())

            for param in sorted(all_params):
                values = []
                for label in labels:
                    config = run_data[label].get('summary', {}).get('config', {}).get(config_type, {})
                    values.append(config.get(param, '-'))

                # Highlight differences
                all_same = all(str(v) == str(values[0]) for v in values)
                marker = "  " if all_same else "⚡"
                values_str = " | ".join([f"{label}: {v}" for label, v in zip(labels, values)])
                print(f"   {marker} {param:<30} {values_str}")
    else:
        print("   (No config data in logs — add config to summary block to enable)")

    # ══════════════════════════════════════════════════════════════════
    # PER-DOCUMENT COMPARISON
    # ══════════════════════════════════════════════════════════════════
    print(f"\n{'='*80}")
    print(f"PER-DOCUMENT COMPARISON")
    print(f"{'='*80}")

    total_pages_per_run = {label: 0 for label in labels}
    total_original_per_run = {label: 0 for label in labels}

    for filename in all_filenames:
        # Get pages for each run
        run_pages = {}
        total_pages = 0
        for label in labels:
            doc_data = all_runs[label].get(filename, {'pages': set(), 'total_pages': 0})
            run_pages[label] = doc_data['pages']
            total_pages = max(total_pages, doc_data['total_pages'])
            total_pages_per_run[label] += len(doc_data['pages'])
            total_original_per_run[label] += doc_data['total_pages']

        # Calculate set operations
        all_page_sets = [run_pages[label] for label in labels]
        common = set.intersection(*all_page_sets) if all(all_page_sets) else set()
        union = set.union(*all_page_sets) if any(all_page_sets) else set()

        # Per-run unique pages (pages only in this run)
        unique_per_run = {}
        for label in labels:
            others = set.union(*[run_pages[l] for l in labels if l != label]) if len(labels) > 1 else set()
            unique_per_run[label] = run_pages[label] - others

        # Check if all runs are identical
        all_identical = all(run_pages[labels[0]] == run_pages[label] for label in labels[1:])

        status = "✅" if all_identical else "🔄"
        print(f"\n{status} {filename} ({total_pages} total pages)")

        # Page counts per run
        counts_str = " | ".join([f"{label}: {len(run_pages[label])}" for label in labels])
        print(f"   Pages selected: {counts_str}")

        if not all_identical:
            print(f"   Common pages: {len(common)}")

            # Show what each run has uniquely
            for label in labels:
                unique = unique_per_run[label]
                if unique:
                    unique_sorted = sorted(unique)
                    if len(unique_sorted) > 15:
                        pages_str = f"{unique_sorted[:15]}... (+{len(unique_sorted)-15} more)"
                    else:
                        pages_str = str(unique_sorted)
                    print(f"   Only in {label}: {pages_str}")

            # Show pages removed/added between consecutive runs
            if len(labels) == 2:
                removed = run_pages[labels[0]] - run_pages[labels[1]]
                added = run_pages[labels[1]] - run_pages[labels[0]]
                if removed:
                    print(f"   ❌ Dropped ({labels[0]}→{labels[1]}): {sorted(removed)}")
                if added:
                    print(f"   ➕ Added ({labels[0]}→{labels[1]}): {sorted(added)}")

    # ══════════════════════════════════════════════════════════════════
    # OVERALL SUMMARY
    # ══════════════════════════════════════════════════════════════════
    print(f"\n{'='*80}")
    print(f"OVERALL SUMMARY")
    print(f"{'='*80}")

    # Summary table
    header = f"{'Metric':<30}" + "".join([f"{label:>15}" for label in labels])
    print(f"\n{header}")
    print(f"{'-'*len(header)}")

    # Total original pages
    row = f"{'Original pages':<30}" + "".join([f"{total_original_per_run[label]:>15}" for label in labels])
    print(row)

    # Total filtered pages
    row = f"{'Filtered pages':<30}" + "".join([f"{total_pages_per_run[label]:>15}" for label in labels])
    print(row)

    # Reduction %
    reductions = []
    for label in labels:
        if total_original_per_run[label] > 0:
            red = (1 - total_pages_per_run[label] / total_original_per_run[label]) * 100
        else:
            red = 0
        reductions.append(red)
    row = f"{'Reduction %':<30}" + "".join([f"{r:>14.1f}%" for r in reductions])
    print(row)

    # Page difference from first run
    if len(labels) >= 2:
        baseline = total_pages_per_run[labels[0]]
        row = f"{'Δ vs {0}':<30}".format(labels[0]) + "".join([
            f"{total_pages_per_run[label] - baseline:>+15}" if label != labels[0] else f"{'—':>15}"
            for label in labels
        ])
        print(row)

        # % change from first run
        row = f"{'Δ% vs {0}':<30}".format(labels[0]) + "".join([
            f"{((total_pages_per_run[label] - baseline) / baseline * 100):>+14.1f}%" if label != labels[0] and baseline > 0 else f"{'—':>15}"
            for label in labels
        ])
        print(row)

    # Files processed
    for label in labels:
        summary = run_data[label].get('summary', {})
        files = summary.get('total_files_processed', 'N/A')
        print(f"\n   {label}: {files} files processed")

    print()
    return {
        'labels': labels,
        'total_filtered': total_pages_per_run,
        'total_original': total_original_per_run,
    }


# ── Batch comparison across multiple companies ──

def compare_all_companies(companies: List[str], log_dirs: List[Path],
                          run_labels: List[str] = None):
    """
    Run compare_pipeline_runs for multiple companies and print a final summary.

    Usage:
        compare_all_companies(TEST_COMPANIES, [
            DRIVE_BASE / "ESG Data" / "filtering_logs_1",
            DRIVE_BASE / "ESG Data" / "filtering_logs_2",
        ], run_labels=["baseline", "tight_v1"])
    """
    if run_labels is None:
        run_labels = [f"Run {i+1}" for i in range(len(log_dirs))]

    all_results = {}
    for company in companies:
        result = compare_pipeline_runs(company, log_dirs, run_labels)
        if result:
            all_results[company] = result

    if not all_results:
        return

    # ── Grand total across all companies ──
    labels = run_labels
    print(f"\n{'='*80}")
    print(f"GRAND TOTAL ACROSS {len(all_results)} COMPANIES")
    print(f"{'='*80}")

    grand_filtered = {label: 0 for label in labels}
    grand_original = {label: 0 for label in labels}

    for company, result in all_results.items():
        for label in labels:
            grand_filtered[label] += result['total_filtered'].get(label, 0)
            grand_original[label] += result['total_original'].get(label, 0)

    header = f"{'Metric':<30}" + "".join([f"{label:>15}" for label in labels])
    print(f"\n{header}")
    print(f"{'-'*len(header)}")

    row = f"{'Total original pages':<30}" + "".join([f"{grand_original[label]:>15}" for label in labels])
    print(row)

    row = f"{'Total filtered pages':<30}" + "".join([f"{grand_filtered[label]:>15}" for label in labels])
    print(row)

    reductions = []
    for label in labels:
        red = (1 - grand_filtered[label] / grand_original[label] * 1.0) * 100 if grand_original[label] > 0 else 0
        reductions.append(red)
    row = f"{'Reduction %':<30}" + "".join([f"{r:>14.1f}%" for r in reductions])
    print(row)

    if len(labels) >= 2:
        baseline = grand_filtered[labels[0]]
        row = f"{'Δ pages vs {0}':<30}".format(labels[0]) + "".join([
            f"{grand_filtered[label] - baseline:>+15}" if label != labels[0] else f"{'—':>15}"
            for label in labels
        ])
        print(row)

    print()


print("✅ Pipeline comparison tools defined")
print("   Usage: compare_pipeline_runs('Company', [log_dir_1, log_dir_2], ['baseline', 'tight'])")
print("   Usage: compare_all_companies(TEST_COMPANIES, [log_dir_1, log_dir_2], ['baseline', 'tight'])")

✅ Pipeline comparison tools defined
   Usage: compare_pipeline_runs('Company', [log_dir_1, log_dir_2], ['baseline', 'tight'])
   Usage: compare_all_companies(TEST_COMPANIES, [log_dir_1, log_dir_2], ['baseline', 'tight'])


In [ ]:
# Single company comparison:
compare_pipeline_runs("Nvidia", [
    DRIVE_BASE / "ESG Data" / "filtering_logs",
    DRIVE_BASE / "ESG Data" / "filtering_logs_1",
], run_labels=["TOC Filter Removed", "baseline"])


PIPELINE RUN COMPARISON: Nvidia

📋 Configuration Comparison:
--------------------------------------------------------------------------------

   DEF14A:
      adjacent_page_threshold        TOC Filter Removed: 0.4 | baseline: 0.4
      api_model                      TOC Filter Removed: claude-haiku-4-5-20251001 | baseline: claude-haiku-4-5-20251001
      max_pages_to_select            TOC Filter Removed: 30 | baseline: 30
      min_category_score             TOC Filter Removed: 1 | baseline: 1
      top_pages_per_category         TOC Filter Removed: 3 | baseline: 3
      use_api_classification         TOC Filter Removed: False | baseline: False

   SUSTAINABILITY:
      adjacent_page_threshold        TOC Filter Removed: 0.4 | baseline: 0.4
      cdp_context_pages              TOC Filter Removed: 1 | baseline: 1
      max_pages_to_select            TOC Filter Removed: 50 | baseline: 50
      min_issue_score                TOC Filter Removed: 2 | baseline: 2
      score_gap_ratio      

{'labels': ['TOC Filter Removed', 'baseline'],
 'total_filtered': {'TOC Filter Removed': 478, 'baseline': 469},
 'total_original': {'TOC Filter Removed': 1207, 'baseline': 1207}}

In [ ]:
# All test companies at once:
# compare_all_companies(TEST_COMPANIES, [
#     DRIVE_BASE / "ESG Data" / "filtering_logs_1",
#     DRIVE_BASE / "ESG Data" / "filtering_logs_2",
#     DRIVE_BASE / "ESG Data" / "filtering_logs_3",
# ], run_labels=["baseline", "tight_v1", "tight_v2"])

### Debug Specific Pages

In [ ]:
# =============================================================================
# PAGE DIAGNOSTIC TOOL
# =============================================================================

def diagnose_sustainability_pages(company: str, filename: str, input_dir: Path,
                                  target_pages: List[int] = None):
    """
    Show detailed scoring for ALL pages (or specific target pages) in a
    sustainability report. Reveals why pages were/weren't selected.

    Args:
        company: Company name
        filename: PDF filename
        input_dir: Input directory
        target_pages: Optional list of page numbers to focus on.
                      If None, shows all pages with any keyword match.

    Usage:
        diagnose_sustainability_pages("Nvidia", "Report_Nvidia_2021_94.pdf",
                                      INPUT_DIR, target_pages=[73, 74, 75])
    """
    pdf_path = Path(input_dir) / company / filename

    if not pdf_path.exists():
        print(f"❌ File not found: {pdf_path}")
        return

    print(f"{'='*80}")
    print(f"PAGE DIAGNOSTICS: {filename}")
    print(f"{'='*80}")

    # Detect doc type
    doc_type = detect_sustainability_doc_type(str(pdf_path), filename)
    print(f"Document type: {doc_type}")

    with pdfplumber.open(str(pdf_path)) as pdf:
        print(f"Total pages: {len(pdf.pages)}")

        pages_to_check = target_pages if target_pages else range(1, len(pdf.pages) + 1)

        for page_num in pages_to_check:
            if page_num < 1 or page_num > len(pdf.pages):
                print(f"\n⚠️ Page {page_num} out of range")
                continue

            page = pdf.pages[page_num - 1]
            text = page.extract_text() or ''
            text_lower = text.lower()
            tables = page.extract_tables()

            print(f"\n{'─'*80}")
            print(f"📄 PAGE {page_num}")
            print(f"{'─'*80}")

            # ── Text preview ──
            preview = text[:300].replace('\n', ' ').strip()
            print(f"Text preview: {preview}...")

            # ── Exclusion check ──
            penalty = get_exclusion_penalty(text_lower, SUSTAINABILITY_EXCLUSION_KEYWORDS)
            matched_exclusions = [kw for kw in SUSTAINABILITY_EXCLUSION_KEYWORDS if kw in text_lower]
            if matched_exclusions:
                print(f"\n⛔ Exclusion matches: {matched_exclusions}")
                print(f"   Penalty: {penalty}")
            else:
                print(f"\n✅ No exclusion matches (penalty: 0)")

            # ── TOC footer check ──
            if 'table of contents' in text_lower:
                print(f"   ℹ️  'table of contents' found in text (likely footer/header)")

            # ── Data evidence ──
            data_count = count_data_numbers(text)
            best_table_quality = 0
            table_count = 0
            if tables:
                table_count = len(tables)
                for t in tables:
                    _, q = is_data_table(t)
                    best_table_quality = max(best_table_quality, q)

            has_data = data_count >= 3 or best_table_quality >= 8
            print(f"\n📊 Data evidence:")
            print(f"   Numbers found: {data_count} (threshold: 3)")
            print(f"   Tables found: {table_count}")
            print(f"   Best table quality: {best_table_quality} (threshold: 8)")
            print(f"   Has data: {'✅ YES' if has_data else '❌ NO'}")

            # ── Year and unit matches ──
            years_found = [y for y in YEAR_KEYWORDS if y in text]
            units_found = [u for u in UNIT_KEYWORDS if u in text_lower]
            print(f"   Years: {years_found}")
            print(f"   Units: {units_found}")

            # ── Per-issue scoring ──
            print(f"\n📋 Issue scores:")
            issue_scores = []

            for issue, keywords in SUSTAINABILITY_ISSUE_KEYWORDS.items():
                matched = [kw for kw in keywords if kw in text_lower]
                if not matched:
                    continue

                # Replicate the scoring logic from _filter_standard_sustainability
                score = len(matched)

                if has_data:
                    score += len(years_found) * 2
                    score += len(units_found) * 3
                    score += best_table_quality
                else:
                    score = min(score, 3)

                score += penalty

                final_score = max(score, 0)
                issue_scores.append((issue, final_score, matched))

                status = "✅" if final_score > 0 else "❌"
                print(f"   {status} {issue:<20} score={final_score:<6} keywords={matched}")

            if not issue_scores:
                print(f"   (no keyword matches on this page)")

            # ── Table indicator scoring ──
            table_matched = [kw for kw in TABLE_INDICATOR_KEYWORDS if kw in text_lower]
            if table_matched:
                if has_data:
                    t_score = len(table_matched) * 5 + best_table_quality
                    t_score += len(years_found) * 3
                    print(f"\n📋 Table indicator: score={t_score}, keywords={table_matched}")
                else:
                    print(f"\n📋 Table indicator: keywords found but no data evidence → not scored")

            # ── Summary for this page ──
            if issue_scores:
                best_issue, best_score, _ = max(issue_scores, key=lambda x: x[1])
                print(f"\n   ⭐ Best issue: {best_issue} (score {best_score})")
            else:
                print(f"\n   ⭐ No issue scores — page would NOT be selected")

    print(f"\n{'='*80}")
    print(f"END DIAGNOSTICS")
    print(f"{'='*80}")


print("✅ Page diagnostic tool defined")
print("   Usage: diagnose_sustainability_pages('Nvidia', 'Report_Nvidia_2021_94.pdf', INPUT_DIR, [73, 74, 75])")

✅ Page diagnostic tool defined
   Usage: diagnose_sustainability_pages('Nvidia', 'Report_Nvidia_2021_94.pdf', INPUT_DIR, [73, 74, 75])


In [ ]:
diagnose_sustainability_pages("Nvidia", "Report_Nvidia_2021_3.pdf", INPUT_DIR, target_pages=[73, 74, 75])

PAGE DIAGNOSTICS: Report_Nvidia_2021_3.pdf
Document type: standard
Total pages: 94

────────────────────────────────────────────────────────────────────────────────
📄 PAGE 73
────────────────────────────────────────────────────────────────────────────────
Text preview: Detailed Gender Data POSITIONS HELD BY WOMEN FY21 FY20 FY19 Outside Directors 25.0% 18.2% 18.2% Executive Officers 40.0% 40.0% 40.0% Leaders 11.3% 10.5% 11.0% Managers 15.9% 16.2% 16.5% In Technical Roles 14.1% 13.6% 13.7% In Global Workforce 19.1% 18.9% 18.9% New Hires 19.7% 18.8% 20.4% For a break...

✅ No exclusion matches (penalty: 0)
   ℹ️  'table of contents' found in text (likely footer/header)

📊 Data evidence:
   Numbers found: 83 (threshold: 3)
   Tables found: 3
   Best table quality: 20 (threshold: 8)
   Has data: ✅ YES
   Years: ['2020', '2021']
   Units: ['tco2e']

📋 Issue scores:
   ✅ carbon_emissions     score=33     keywords=['scope 1', 'scope 2', 'tco2e', 'co2e', 'greenhouse gas', 'market-based']
   ✅ e

In [ ]:
# =============================================================================
# DEF 14A PAGE DIAGNOSTIC TOOL
# =============================================================================

def diagnose_def14a_pages(company: str, filename: str, input_dir: Path,
                          target_pages: List[int] = None):
    """
    Show detailed regex matching for ALL pages (or specific target pages)
    in a DEF 14A filing. Reveals why pages were/weren't selected.

    Args:
        company: Company name
        filename: PDF filename
        input_dir: Input directory
        target_pages: Optional list of page numbers to focus on.
                      If None, shows all pages with any regex match.

    Usage:
        diagnose_def14a_pages("Nvidia", "SEC_DEF_14A_Nvidia_2020.pdf",
                              INPUT_DIR, target_pages=[3, 4, 5, 40, 75])
    """
    pdf_path = Path(input_dir) / company / filename

    if not pdf_path.exists():
        print(f"❌ File not found: {pdf_path}")
        return

    # Compile patterns
    compiled = {}
    for category, data in SEC_DEF14A_PATTERNS.items():
        compiled[category] = {
            'patterns': [(p, re.compile(p, re.IGNORECASE)) for p in data['patterns']],
            'weight': data['weight'],
        }

    doc = fitz.open(str(pdf_path))
    total_pages = len(doc)

    print(f"{'='*80}")
    print(f"DEF 14A DIAGNOSTICS: {filename}")
    print(f"{'='*80}")
    print(f"Total pages: {total_pages}")

    # First pass: score all pages to know the competition
    all_page_scores = {}  # {page_num: {category: score}}
    for page_idx in range(total_pages):
        page = doc[page_idx]
        text = page.get_text()
        page_num = page_idx + 1
        all_page_scores[page_num] = {}

        for category, cat_data in compiled.items():
            matched = [(p_str, p_re) for p_str, p_re in cat_data['patterns'] if p_re.search(text)]
            if matched:
                score = len(matched) * cat_data['weight']
                all_page_scores[page_num][category] = {
                    'score': score,
                    'match_count': len(matched),
                }

    # Show top pages per category (the competition)
    print(f"\n📊 Current Top Pages Per Category (what selected pages compete against):")
    print(f"{'-'*80}")

    top_per_category = {}
    for category in SEC_DEF14A_PATTERNS.keys():
        cat_scores = []
        for pn, cats in all_page_scores.items():
            if category in cats:
                cat_scores.append((pn, cats[category]['score']))
        cat_scores.sort(key=lambda x: -x[1])
        top_per_category[category] = cat_scores

        top_3 = cat_scores[:3]
        weight = SEC_DEF14A_PATTERNS[category]['weight']
        top_str = ', '.join([f"p{pn}({s})" for pn, s in top_3])
        print(f"   {category:<15} [w={weight}] → {top_str:<45} ({len(cat_scores)} candidates)")

    # Detailed diagnostics for target pages
    pages_to_check = target_pages if target_pages else [
        pn for pn in range(1, total_pages + 1) if all_page_scores.get(pn, {})
    ]

    if target_pages:
        # Also include pages with no matches if specifically requested
        pages_to_check = target_pages

    for page_num in pages_to_check:
        if page_num < 1 or page_num > total_pages:
            print(f"\n⚠️ Page {page_num} out of range")
            continue

        page = doc[page_num - 1]
        text = page.get_text()
        text_lower = text.lower()

        print(f"\n{'─'*80}")
        print(f"📄 PAGE {page_num}")
        print(f"{'─'*80}")

        # Text preview
        preview = text[:400].replace('\n', ' ').strip()
        print(f"Preview: {preview}...")

        # Exclusion check
        penalty = get_exclusion_penalty(text_lower, DEF14A_EXCLUSION_KEYWORDS)
        matched_exclusions = [kw for kw in DEF14A_EXCLUSION_KEYWORDS if kw in text_lower]
        if matched_exclusions:
            print(f"\n⛔ Exclusion matches: {matched_exclusions} (penalty: {penalty})")
        else:
            print(f"\n✅ No exclusion matches (penalty: 0)")

        # Regex matches per category
        print(f"\n📋 Category Matches:")
        page_has_matches = False

        for category, cat_data in compiled.items():
            weight = cat_data['weight']
            matched = []
            unmatched = []

            for p_str, p_re in cat_data['patterns']:
                m = p_re.search(text)
                if m:
                    matched.append((p_str[:60], m.group()[:60]))
                else:
                    unmatched.append(p_str[:60])

            if matched:
                page_has_matches = True
                score = len(matched) * weight + penalty
                # Check if this page would make top 3
                rank = 1
                for pn, s in top_per_category.get(category, []):
                    if pn == page_num:
                        break
                    rank += 1

                rank_str = f"rank #{rank}" if rank <= 3 else f"rank #{rank} (NOT in top 3)"
                selected = "✅" if rank <= 3 else "❌"

                print(f"\n   {selected} {category} — score={score}, {len(matched)} matches, weight={weight}, {rank_str}")
                for p_str, match_text in matched:
                    print(f"      ✅ '{p_str}'")
                    print(f"         → matched: '{match_text}'")
            else:
                # Only show unmatched categories if target pages specified
                if target_pages:
                    print(f"\n   ❌ {category} — no matches")

        if not page_has_matches:
            print(f"   (no regex matches on this page)")

            # Check for potentially useful keywords that aren't in our patterns
            interesting = [
                'proxy summary', 'summary', 'director', 'board', 'independent',
                'compensation', 'diversity', 'female', 'women', 'pay ratio',
                'approval', 'vote', 'say on pay', 'say-on-pay', 'ethics',
                'ownership', 'proposal',
            ]
            found = [kw for kw in interesting if kw in text_lower]
            if found:
                print(f"   💡 Unmatched keywords found: {found}")

    doc.close()

    print(f"\n{'='*80}")
    print(f"END DIAGNOSTICS")
    print(f"{'='*80}")


print("✅ DEF 14A diagnostic tool defined")
print("   Usage: diagnose_def14a_pages('Nvidia', 'SEC_DEF_14A_Nvidia_2020.pdf', INPUT_DIR, [3, 4, 5])")

✅ DEF 14A diagnostic tool defined
   Usage: diagnose_def14a_pages('Nvidia', 'SEC_DEF_14A_Nvidia_2020.pdf', INPUT_DIR, [3, 4, 5])


In [ ]:
# Check the proxy summary pages and any other missed pages
diagnose_def14a_pages("Nvidia", "SEC_DEF_14A_Nvidia_2020.pdf", INPUT_DIR,
                      target_pages= [7, 11, 50])

DEF 14A DIAGNOSTICS: SEC_DEF_14A_Nvidia_2020.pdf
Total pages: 148

📊 Current Top Pages Per Category (what selected pages compete against):
--------------------------------------------------------------------------------
   board           [w=3] → p5(12), p18(12), p31(12)                      (35 candidates)
   diversity       [w=4] → p19(4)                                        (1 candidates)
   meetings        [w=3] → p5(6), p31(6), p13(3)                         (9 candidates)
   pay_ratio       [w=6] → p68(30), p5(6)                                (2 candidates)
   say_on_pay      [w=5] → p42(20), p9(15), p5(10)                       (10 candidates)
   ethics          [w=2] → p30(2), p33(2), p35(2)                        (5 candidates)
   ownership       [w=2] → p40(6), p41(4), p82(4)                        (24 candidates)

────────────────────────────────────────────────────────────────────────────────
📄 PAGE 7
──────────────────────────────────────────────────────────────────────

### Scan Exclusion Keywords

In [ ]:
# =============================================================================
# EXCLUSION KEYWORD SCANNER
# =============================================================================

def scan_exclusion_keywords(company: str, filename: str, input_dir: Path,
                            doc_type: str = "sustainability"):
    """
    Scan every page of a document for exclusion keyword matches.
    Shows which pages are penalized and by how much.

    Args:
        company: Company name
        filename: PDF filename
        input_dir: Input directory
        doc_type: "sustainability" or "def14a"

    Usage:
        scan_exclusion_keywords("Nvidia", "Report_Nvidia_2021_3.pdf", INPUT_DIR)
        scan_exclusion_keywords("Nvidia", "SEC_DEF_14A_Nvidia_2021.pdf", INPUT_DIR, "def14a")
    """
    pdf_path = Path(input_dir) / company / filename
    if not pdf_path.exists():
        print(f"❌ File not found: {pdf_path}")
        return

    if doc_type == "def14a":
        exclusion_list = DEF14A_EXCLUSION_KEYWORDS
    else:
        exclusion_list = SUSTAINABILITY_EXCLUSION_KEYWORDS

    print(f"{'='*80}")
    print(f"EXCLUSION SCAN: {filename}")
    print(f"Document type: {doc_type}")
    print(f"Exclusion keywords: {len(exclusion_list)}")
    print(f"{'='*80}")

    import fitz
    doc = fitz.open(str(pdf_path))
    total_pages = len(doc)

    affected_pages = []
    keyword_frequency = {}  # keyword -> count of pages it appears on

    for page_idx in range(total_pages):
        page = doc[page_idx]
        text = page.get_text().lower()
        page_num = page_idx + 1

        matched = []
        for kw in exclusion_list:
            if kw in text:
                matched.append(kw)
                keyword_frequency[kw] = keyword_frequency.get(kw, 0) + 1

        if matched:
            penalty = get_exclusion_penalty(text, exclusion_list)

            # Find short context for each match
            contexts = []
            for kw in matched:
                idx = text.find(kw)
                start = max(0, idx - 30)
                end = min(len(text), idx + len(kw) + 30)
                snippet = text[start:end].replace('\n', ' ').strip()
                contexts.append(f'"{snippet}"')

            affected_pages.append({
                'page_num': page_num,
                'penalty': penalty,
                'keywords': matched,
                'contexts': contexts,
                'would_exclude': penalty <= -100,
            })

    doc.close()

    # ── Results ──
    excluded = [p for p in affected_pages if p['would_exclude']]
    penalized = [p for p in affected_pages if not p['would_exclude']]

    print(f"\nTotal pages: {total_pages}")
    print(f"Pages with exclusion matches: {len(affected_pages)}")
    print(f"  Hard excluded (penalty ≤ -100): {len(excluded)}")
    print(f"  Penalized (penalty > -100): {len(penalized)}")
    print(f"  Clean (no matches): {total_pages - len(affected_pages)}")

    # Keyword frequency
    print(f"\n📊 Keyword Frequency (how many pages each keyword appears on):")
    print(f"{'-'*60}")
    for kw, count in sorted(keyword_frequency.items(), key=lambda x: -x[1]):
        pct = count / total_pages * 100
        flag = " ⚠️ APPEARS ON >50% OF PAGES" if pct > 50 else ""
        print(f"   '{kw}': {count}/{total_pages} pages ({pct:.0f}%){flag}")

    # Hard excluded pages
    if excluded:
        print(f"\n⛔ Hard Excluded Pages (penalty ≤ -100):")
        print(f"{'-'*60}")
        for p in excluded:
            print(f"   Page {p['page_num']}: penalty={p['penalty']}")
            for kw, ctx in zip(p['keywords'], p['contexts']):
                print(f"      '{kw}' → {ctx}")

    # Penalized pages
    if penalized:
        print(f"\n⚠️ Penalized Pages:")
        print(f"{'-'*60}")
        for p in penalized:
            print(f"   Page {p['page_num']}: penalty={p['penalty']}, keywords={p['keywords']}")

    # Summary of affected page numbers
    if affected_pages:
        all_affected = sorted([p['page_num'] for p in affected_pages])
        excluded_nums = sorted([p['page_num'] for p in excluded])
        print(f"\nAll affected pages: {all_affected}")
        if excluded_nums:
            print(f"Hard excluded pages: {excluded_nums}")

    return affected_pages


print("✅ Exclusion keyword scanner defined")
print("   Usage: scan_exclusion_keywords('Nvidia', 'Report_Nvidia_2021_3.pdf', INPUT_DIR)")

✅ Exclusion keyword scanner defined
   Usage: scan_exclusion_keywords('Nvidia', 'Report_Nvidia_2021_3.pdf', INPUT_DIR)


In [ ]:
scan_exclusion_keywords("Nvidia", "Report_Nvidia_2021_3.pdf", INPUT_DIR)

EXCLUSION SCAN: Report_Nvidia_2021_3.pdf
Document type: sustainability
Exclusion keywords: 16

Total pages: 94
Pages with exclusion matches: 8
  Hard excluded (penalty ≤ -100): 3
  Penalized (penalty > -100): 5
  Clean (no matches): 86

📊 Keyword Frequency (how many pages each keyword appears on):
------------------------------------------------------------
   'about this report': 4/94 pages (4%)
   'letter from our ceo': 3/94 pages (3%)
   'un sustainable development goals': 3/94 pages (3%)
   'forward-looking statements': 1/94 pages (1%)

⛔ Hard Excluded Pages (penalty ≤ -100):
------------------------------------------------------------
   Page 78: penalty=-100
      'about this report' → "anagement and product quality about this report 102-13 membership of associat"
   Page 80: penalty=-100
      'about this report' → "ting. 102-50 reporting period about this report 102-51 date of most recent re"
   Page 94: penalty=-100
      'about this report' → "2021 nvidia csr report  about th

[{'page_num': 2,
  'penalty': -45,
  'keywords': ['about this report',
   'letter from our ceo',
   'un sustainable development goals'],
  'contexts': ['"tainable development goals 91 about this report 01 executive messages letter"',
   '"report 01 executive messages letter from our ceo 02 q&a with our cfo 05 02 pri"',
   '"6 sasb index 87 tcfd index 90 un sustainable development goals 91 about this report 01 execu"'],
  'would_exclude': False},
 {'page_num': 3,
  'penalty': -15,
  'keywords': ['letter from our ceo'],
  'contexts': ['"ontents 01 executive messages letter from our ceo each year we share in our cor"'],
  'would_exclude': False},
 {'page_num': 48,
  'penalty': -15,
  'keywords': ['un sustainable development goals'],
  'contexts': ['"ore about our support for the un sustainable development goals can be found here. what it co"'],
  'would_exclude': False},
 {'page_num': 78,
  'penalty': -100,
  'keywords': ['about this report'],
  'contexts': ['"anagement and product qualit